# The Hidden Advantage of Mask Resampling

Reproduce the paper figures from the saved results. Run the setup cell first, then any figure cell or **Run All**. PDFs are written to the repository's single `plots/` directory. No training or external data access occurs here.

All 18 figures appear in the manuscript. See [the figure guide](../docs/figures.md) and [data guide](../docs/data.md). Error bars and bands retain the source plots' sample standard deviation (SD); single-seed comparisons have no seed error bars. Numerical statuses are preserved.


## Setup and publication style

The setup locates the repository from either its root or the `notebooks/` directory. Set `USE_TEX=False` to use Matplotlib without TeX.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "manifest.json").is_file()
             and (p / "notebooks" / "figures.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from this repository or its notebooks/ directory.")
DATA_ROOT = ROOT / "data"
PLOT_DIR = ROOT / "plots"
PLOT_DIR.mkdir(exist_ok=True)

# Verify all packaged inputs before plotting. No external files are used.
input_manifest = json.loads((DATA_ROOT / "manifest.json").read_text())
for entry in input_manifest["files"]:
    path = ROOT / entry["path"]
    if hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
        raise ValueError(f"Data checksum mismatch: {entry['path']}")

# Optional TeX rendering. False works without a TeX installation.
USE_TEX = shutil.which("latex") is not None and shutil.which("dvipng") is not None

def save_pdf(figure, filename, **kwargs):
    """Save PDFs only in plots/, without personal or timestamp metadata."""
    if Path(filename).name != filename or not filename.endswith(".pdf"):
        raise ValueError("Use a PDF filename, without subdirectories.")
    metadata = {"Author": None, "Creator": "Matplotlib", "Title": filename[:-4],
                "Subject": None, "Keywords": None, "CreationDate": None, "ModDate": None}
    figure.savefig(PLOT_DIR / filename, metadata=metadata, **kwargs)
    print(f"Saved plots/{filename}")

# All multiseed uncertainty bars and bands use sample standard deviation (SD).
# Global style controls: 1.0 restores the original text/marker sizes.
FONT_SCALE = 1.15
SHOW_DATASET_NAMES = False  # False hides dataset/model-setting suffixes in titles.
MARKER_SCALE = 1.25  # Linear marker size; scatter areas use MARKER_SCALE ** 2.
if FONT_SCALE <= 0 or MARKER_SCALE <= 0:
    raise ValueError("Style scale factors must be positive.")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "text.usetex": USE_TEX,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"] if USE_TEX else ["DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 12 * FONT_SCALE
})

# Shared publication palette. Change only K_COLORMAP_NAME to recolor every
# finite-K series together with the static and dynamic endpoint colors.
K_COLORMAP_NAME = "cividis"
# Use only this interval of the colormap (positions between 0 and 1).
# For cividis, lower MAX to reduce the bright yellow used for static.
K_COLORMAP_MIN = 0.1
K_COLORMAP_MAX = 0.85
if not (0.0 <= K_COLORMAP_MIN < K_COLORMAP_MAX <= 1.0):
    raise ValueError("Colormap limits must satisfy 0 <= MIN < MAX <= 1.")
K_COLORMAP = plt.get_cmap(K_COLORMAP_NAME)


def _publication_color_luminance(color):
    rgb = np.asarray(color[:3], dtype=float)
    linear_rgb = np.where(
        rgb <= 0.04045,
        rgb / 12.92,
        ((rgb + 0.055) / 1.055) ** 2.4,
    )
    return float(np.dot(linear_rgb, (0.2126, 0.7152, 0.0722)))


_K_COLORMAP_ENDPOINTS = (K_COLORMAP_MIN, K_COLORMAP_MAX)
K_COLORMAP_LIGHT_POSITION = max(
    _K_COLORMAP_ENDPOINTS,
    key=lambda position: _publication_color_luminance(K_COLORMAP(position)),
)
K_COLORMAP_DARK_POSITION = min(
    _K_COLORMAP_ENDPOINTS,
    key=lambda position: _publication_color_luminance(K_COLORMAP(position)),
)


def publication_k_colors(k_order):
    k_order = tuple(k_order)
    if not k_order:
        return {}
    positions = np.linspace(
        K_COLORMAP_LIGHT_POSITION,
        K_COLORMAP_DARK_POSITION,
        len(k_order),
    )
    return {
        key: K_COLORMAP(float(position))
        for key, position in zip(k_order, positions)
    }


MASKING_COLORS = publication_k_colors(("static", "dynamic"))
STATIC_MASKING_COLOR = MASKING_COLORS["static"]
DYNAMIC_MASKING_COLOR = MASKING_COLORS["dynamic"]
NO_MASKING_COLOR = "#74212B"
STANDARD_AUGMENTATION_COLOR = "#7A5195"
BAYES_OPTIMAL_COLOR = "#366A3CFC"
AMP_COLOR = BAYES_OPTIMAL_COLOR

# Shared palette for the plotting cells.
colors_paper = {
    "dynamic": DYNAMIC_MASKING_COLOR,
    "static": STATIC_MASKING_COLOR,
    "linear": "#44AA99",
    "relu": "#E69F00",
    "he2_phe1": "#838181",
}

# Activation colors (separate from the colormap encoding the number of masks).
ACTIVATION_COLORS = {
    "linear": colors_paper["linear"],
    "relu": colors_paper["relu"],
    "elu": "#4477AA",
    "tanh": "#AA4499",
}

# Shared dimensions and typography for two-panel publication figures.
TWO_PANEL_STYLE = {
    "figsize": (7.0, 1.8),
    "title": 9 * FONT_SCALE,
    "label": 8 * FONT_SCALE,
    "tick": 7 * FONT_SCALE,
    "legend": 6.3 * FONT_SCALE,
}


def publication_title(model, dataset=None, *, multiline=False):
    """Optionally append the dataset/setting, preserving the panel/model name."""
    if not SHOW_DATASET_NAMES or not dataset:
        return model
    separator = "\n" if multiline else " "
    return f"{model}{separator}({dataset})"


## Figure 1: Mask resampling across synthetic, CNN and ViT models

Synthetic panels use the bounded Beta law, beta=1, alpha=8, d=2000, 30 ERM seeds and ridge 1e-4. The red synthetic reference is the analytical asymptotic PCA chance value 1/2. CNN accuracy is test top-1; ViT accuracy is validation top-5.

Output: `plots/figure_01_mask_resampling.pdf`.


In [ ]:
# Uncertainty: plotted error bars and shaded bands are sample standard deviation (SD).
# ============================================================
# ============================================================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# All three panels use the same self-contained linear-probe snapshots as
# the corresponding publication figures in this notebook.

spiked_data_dir = DATA_ROOT / 'synthetic/masking_ratio'
spiked_theory_path = spiked_data_dir / "generic_k_downstream_theory.csv"
spiked_endpoints_path = spiked_data_dir / "downstream_probe_per_seed.csv"
spiked_theory = pd.read_csv(spiked_theory_path)
spiked_endpoints_raw = pd.read_csv(spiked_endpoints_path)
# Asymptotic PCA reference for the same bounded law: beta*m_+ = 8/9 < 1.
# The paper's bounded-noise PCA result implies chance downstream accuracy.
spiked_pca_asymptotic_accuracy = 0.5
# CNN panel uses only the consolidated linear-probe comparison snapshot.
cnn_data_path = DATA_ROOT / 'vision/cnn/masking_ratio/comparison_plot_data.csv'
cnn_data = pd.read_csv(cnn_data_path)
vit_data_path = DATA_ROOT / 'vision/vit/masking_ratio/linear_probe_top5_accuracy_mean_across_initialization_seeds.csv'
vit_data = pd.read_csv(vit_data_path)


def require_columns(frame, required, name):
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")


require_columns(
    spiked_theory,
    {"mask_protocol", "mask_count", "rho", "status", "downstream_test_accuracy"},
    "generic_k_downstream_theory.csv",
)
require_columns(
    spiked_endpoints_raw,
    {"activation", "method", "rho", "seed", "probe_test_accuracy"},
    "downstream_probe_per_seed.csv",
)
require_columns(
    cnn_data,
    {"method", "rho", "n_independent_pretraining_and_split_seeds",
     "test_accuracy_mean", "test_accuracy_std"},
    cnn_data_path.name,
)
if cnn_data.empty or cnn_data.duplicated(["method", "rho"]).any():
    raise ValueError("Expected unique CNN rows per method and rho.")
cnn_methods = {
    "static": "K=1, no image augmentation",
    "dynamic": r"Dynamic masking ($K\to\infty$, step-matched)",
}
require_columns(
    vit_data,
    {"masking_mode", "rho", "mean_top5_accuracy", "sample_std_top5_accuracy", "number_of_seeds", "plot_element"},
    vit_data_path.name,
)

spiked_activation = "linear"
spiked_endpoint_methods = {
    "static": "static_mask_mae_adam",
    "dynamic": "dynamic_mask_mae_adam",
}
spiked_endpoints = (
    spiked_endpoints_raw.loc[
        spiked_endpoints_raw["activation"].eq(spiked_activation)
        & spiked_endpoints_raw["method"].isin(spiked_endpoint_methods.values())
    ]
    .groupby(["method", "rho"], as_index=False)
    .agg(
        num_seeds=("seed", "nunique"),
        probe_test_accuracy_mean=("probe_test_accuracy", "mean"),
        probe_test_accuracy_std=("probe_test_accuracy", "std"),
    )
)

colors = {"static": colors_paper["static"], "dynamic": colors_paper["dynamic"]}
markers = {"static": "o", "dynamic": "o"}
figure1_double_column_width = 7.0  # inches
figure1_original_height = 1.5  # inches
figure1_v2_height = 1.8  # inches, matching Figures 3 and 4 v2
figure1_scale = figure1_double_column_width / (17.5 / 1.5)
figure1_legend_fontsize = 6.3 * FONT_SCALE


def figure1_no_masking(ax, mean, sd=None):
    """No-masking mean line with a translucent +/- SD band, without caps."""
    if sd is not None:
        ax.axhspan(
            mean - sd, mean + sd,
            color=NO_MASKING_COLOR, alpha=0.6, linewidth=0, zorder=1,
        )
    ax.axhline(
        mean, color=NO_MASKING_COLOR, linewidth=2.3 * figure1_scale, alpha=1.0,
        zorder=2, label="Unmasked reconstruction",
    )

fig, axes = plt.subplots(
    1,
    3,
    figsize=(figure1_double_column_width, figure1_original_height),
    constrained_layout=True,
)

# (a) Spiked model: the same linear-probe data used by Figure 4 v3.
ax = axes[0]
theory_series = (
    ("static", "Static masking", "finite quenched masks", 1.0),
    ("dynamic", "Dynamic masking", "annealed/dynamic", None),
)
for schedule, label, mask_protocol, mask_count in theory_series:
    theory_mask = spiked_theory["mask_protocol"].eq(mask_protocol)
    if mask_count is not None:
        theory_mask &= spiked_theory["mask_count"].eq(mask_count)
    group = spiked_theory.loc[
        theory_mask & spiked_theory["status"].eq("converged")
    ].sort_values("rho")
    if group.empty:
        raise ValueError(f"No converged spiked-model theory rows found for {label}.")
    ax.plot(
        group["rho"],
        group["downstream_test_accuracy"],
        color=colors[schedule],
        linewidth=2.3 * figure1_scale,
        label=label,
    )

for schedule, method in spiked_endpoint_methods.items():
    group = spiked_endpoints.loc[
        spiked_endpoints["method"].eq(method)
    ].sort_values("rho")
    if group.empty:
        raise ValueError(f"No linear-probe endpoint rows found for {method!r}.")
    spiked_std = group["probe_test_accuracy_std"].to_numpy(float)
    if not np.isfinite(spiked_std).all():
        raise ValueError(f"Missing shallow-autoencoder standard deviations for {method!r}.")
    if (group["num_seeds"].to_numpy(int) < 2).any():
        raise ValueError(f"Shallow-autoencoder standard deviations require at least two seeds for {method!r}.")
    ax.errorbar(
        group["rho"],
        group["probe_test_accuracy_mean"],
        yerr=spiked_std,  # SD across seeds.
        fmt=markers[schedule],
        linestyle="none",
        color=colors[schedule],
        markerfacecolor=colors[schedule],
        markeredgecolor=colors[schedule],
        markeredgewidth=1.4 * figure1_scale,
        markersize=5.5 * figure1_scale * MARKER_SCALE,
        capsize=2.5 * figure1_scale,
        zorder=4,
    )

# Theoretical limit, with no empirical uncertainty band.
figure1_no_masking(ax, spiked_pca_asymptotic_accuracy)
ax.set_title(publication_title("(a) Shallow Autoencoder", "Spiked model", multiline=True), fontsize=14 * figure1_scale * FONT_SCALE)
ax.set_ylabel("Classification accuracy", fontsize=12 * figure1_scale * FONT_SCALE)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.30, 1.00)

# (b) CNN: masking curves and both references from comparison_plot_data.csv.
ax = axes[1]
for schedule, label in (("static", "Static masking"), ("dynamic", "Dynamic masking")):
    group = cnn_data.loc[cnn_data["method"].eq(cnn_methods[schedule])].sort_values("rho")
    group = group.loc[~(np.isclose(group["rho"], 0.05) | np.isclose(group["rho"], 0.95))]
    if group.empty:
        raise ValueError(f"No CNN rows found for schedule={schedule!r}.")
    cnn_std = group["test_accuracy_std"].to_numpy(float)
    if not np.isfinite(cnn_std).all():
        raise ValueError(f"Missing CNN standard deviations for schedule={schedule!r}.")
    if (group["n_independent_pretraining_and_split_seeds"].to_numpy(int) < 2).any():
        raise ValueError(f"CNN standard deviations require at least two seeds for {schedule!r}.")
    ax.errorbar(
        group["rho"],
        group["test_accuracy_mean"],
        yerr=cnn_std,  # SD.
        color=colors[schedule],
        marker=markers[schedule],
        linestyle="none",
        markerfacecolor=colors[schedule],
        markeredgecolor=colors[schedule],
        markersize=5.5 * figure1_scale * MARKER_SCALE,
        capsize=3.5 * figure1_scale,
        elinewidth=1.3 * figure1_scale,
        capthick=1.3 * figure1_scale,
        zorder=4,
        label=label,
    )

cnn_reference_styles = (
    ("Full reconstruction", NO_MASKING_COLOR, "Unmasked reconstruction"),
    ("Dynamic masking + dynamic U-MAE image augmentation",
     STANDARD_AUGMENTATION_COLOR, "Cropping/flipping + dynamic masking"),
)
for method, color, label in cnn_reference_styles:
    row = cnn_data.loc[cnn_data["method"].eq(method)]
    if len(row) != 1:
        raise ValueError(f"Expected one CNN reference row for {method!r}.")
    mean = float(row.iloc[0]["test_accuracy_mean"])
    std = float(row.iloc[0]["test_accuracy_std"])
    seeds = int(row.iloc[0]["n_independent_pretraining_and_split_seeds"])
    if not np.isfinite([mean, std]).all() or std < 0 or seeds < 2:
        raise ValueError("CNN reference requires a finite mean and sample SD.")
    if label == "Unmasked reconstruction":
        figure1_no_masking(ax, mean, std)
    else:
        ax.axhspan(mean - std, mean + std, color=color, alpha=0.6,
                   linewidth=0, zorder=1)
        ax.axhline(mean, color=color, linewidth=2.3 * figure1_scale,
                   alpha=1.0, zorder=2, label=label)

ax.set_title(publication_title("(b) CNN Autoencoder", "CIFAR-10", multiline=True), fontsize=14 * figure1_scale * FONT_SCALE)
#ax.set_ylabel("Linear-probe test accuracy", fontsize=12)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.30, 0.50)

# (c) ViT: ImageNet-100 top-5 linear-probe validation accuracy.
# Mean and sample SD are provided across initialization seeds in the CSV.
ax = axes[2]
for schedule, label in (("static", "Static masking"), ("dynamic", "Dynamic masking")):
    group = vit_data.loc[
        vit_data["masking_mode"].eq(schedule)
        & vit_data["plot_element"].eq("masking_curve")
    ].sort_values("rho")
    if group.empty or group["rho"].duplicated().any():
        raise ValueError(f"Expected one ViT top-5 row per rho for {schedule!r}.")
    vit_mean = group["mean_top5_accuracy"].to_numpy(float)
    vit_std = group["sample_std_top5_accuracy"].to_numpy(float)
    if not (np.isfinite(vit_mean).all() and np.isfinite(vit_std).all()):
        raise ValueError(f"Missing ViT top-5 mean or standard deviation for {schedule!r}.")
    if (vit_std < 0).any() or (group["number_of_seeds"] < 2).any():
        raise ValueError(f"ViT sample SD requires at least two seeds for {schedule!r}.")
    ax.errorbar(
        group["rho"],
        vit_mean,
        yerr=vit_std,
        color=colors[schedule],
        marker=markers[schedule],
        linestyle="none",
        markerfacecolor=colors[schedule],
        markeredgecolor=colors[schedule],
        markersize=5.5 * figure1_scale * MARKER_SCALE,
        capsize=3.5 * figure1_scale,
        elinewidth=1.3 * figure1_scale,
        capthick=1.3 * figure1_scale,
        zorder=4,
        label=label,
    )

vit_full_reconstruction = vit_data.loc[
    vit_data["masking_mode"].eq("full_reconstruction")
    & vit_data["plot_element"].eq("horizontal_reference")
]
# This single-seed reference is repeated at each rho; draw it only once.
vit_reference_values = vit_full_reconstruction["mean_top5_accuracy"].to_numpy(float)
if (
    len(vit_reference_values) == 0
    or not np.isfinite(vit_reference_values).all()
    or np.ptp(vit_reference_values) > 1e-10
):
    raise ValueError("Expected a constant ViT top-5 full-reconstruction reference.")
figure1_no_masking(ax, float(vit_reference_values[0]))
# Standard augmentation: one dynamic-mask run at rho=0.75, shown as a reference.
vit_standard = vit_data.loc[
    vit_data["augmentation"].eq("standard")
    & vit_data["plot_element"].eq("standard_geometry_horizontal_reference")
]
if len(vit_standard) != 1:
    raise ValueError("Expected one standard-augmentation ViT reference row.")
vit_standard_mean = float(vit_standard.iloc[0]["mean_top5_accuracy"])
if not np.isfinite(vit_standard_mean):
    raise ValueError("Standard-augmentation ViT accuracy must be finite.")
ax.axhline(
    vit_standard_mean, color=STANDARD_AUGMENTATION_COLOR,
    linewidth=2.3 * figure1_scale, alpha=1.0, zorder=2,
    label="Cropping/flipping + dynamic masking",
)
# No uncertainty band: a single seed does not estimate across-seed variability.
ax.set_title(publication_title("(c) ViT-Base", "ImageNet-100", multiline=True), fontsize=14 * figure1_scale * FONT_SCALE)
#ax.set_ylabel("Top-5 accuracy", fontsize=12 * figure1_scale * FONT_SCALE)
ax.set_xlim(0.0, 1.0)

for ax in axes:
    ax.set_xlabel(r"Masking ratio $\rho$", fontsize=12 * figure1_scale * FONT_SCALE)
    #ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.65)
    ax.tick_params(axis="both", labelsize=10 * figure1_scale * FONT_SCALE)
    ax.set_xlim(0,1)
    #ax.legend(loc="best", fontsize=8, frameon=True, framealpha=0.95)
    
figure1_legend_order = ("Dynamic masking", "Static masking", "Unmasked reconstruction", "Cropping/flipping + dynamic masking")
figure1_handles, figure1_labels = axes[2].get_legend_handles_labels()
figure1_handle_by_label = dict(zip(figure1_labels, figure1_handles))
missing_figure1_labels = set(figure1_legend_order) - set(figure1_handle_by_label)
if missing_figure1_labels:
    raise ValueError(f"Missing Figure 1 legend entries: {sorted(missing_figure1_labels)}")
# Shared, frameless legend below the panels, matching Figure 3.
fig.legend(
    [figure1_handle_by_label[label] for label in figure1_legend_order],
    figure1_legend_order,
    loc="upper center", bbox_to_anchor=(0.5, 0.04),
    ncol=4,
    fontsize=figure1_legend_fontsize,
    frameon=False,
    handlelength=1.3,
    handletextpad=0.35,
    columnspacing=0.8,
)
#axes[1].legend(loc="best", fontsize=11, frameon=True, framealpha=0.95)

axes[0].set_ylim(0.45,0.85)
axes[1].set_ylim(0.39, 0.65)
axes[2].set_ylim(0.35, 0.8)

fig.set_size_inches(figure1_double_column_width, figure1_original_height, forward=True)
# Preserve the original tick positions independently of text scaling.
from matplotlib.ticker import FixedLocator
for axis, ticks in zip(axes, ([0.4, 0.5, 0.6, 0.7, 0.8, 0.9], [0.3, 0.4, 0.5, 0.6, 0.7], [0.3, 0.4, 0.5, 0.6, 0.7, 0.8])):
    axis.xaxis.set_major_locator(FixedLocator([0, 0.2, 0.4, 0.6, 0.8, 1]))
    axis.yaxis.set_major_locator(FixedLocator(ticks))
save_pdf(plt.gcf(), 'figure_01_mask_resampling.pdf', dpi=300, bbox_inches='tight')

plt.show()


## Figure 2: Recovery under homogeneous, bounded and unbounded noise

The three variance laws and method-specific dimensions/seed counts are recorded in the CSV. AMP is teacher-initialized with p=0.05. The unbounded PCA zero line is an analytical reference; saved failed solver statuses remain in the data.

Output: `plots/figure_02_noise_laws.pdf`.


In [ ]:
# Uncertainty: plotted error bars and shaded bands are sample standard deviation (SD).
# ============================================================
# ============================================================
# Everything is plotted here from this one CSV; no external plotting scripts.
from pathlib import Path
from matplotlib.ticker import FuncFormatter, NullFormatter

figure4_csv = DATA_ROOT / 'synthetic/three_noise_laws/three_noise_oracle_amp_plant0p05.csv'
figure4_data = pd.read_csv(figure4_csv)
figure4_required = {
    "panel", "kind", "method", "alpha", "cosine", "sd", "sem",
    "num_seeds", "status", "zero_imputed", "d", "beta", "rho",
}
if not figure4_required.issubset(figure4_data.columns):
    raise ValueError(f"Missing columns: {sorted(figure4_required - set(figure4_data.columns))}")
if figure4_data.duplicated(["panel", "kind", "method", "alpha"]).any():
    raise ValueError("Duplicate Figure 4 series/alpha rows.")
figure4_data["zero_imputed"] = (
    figure4_data["zero_imputed"].astype(str).str.lower().eq("true")
)
figure4_sim = figure4_data.loc[figure4_data["kind"].eq("simulation")]
if not np.isfinite(figure4_sim[["alpha", "cosine", "sd"]].to_numpy(float)).all():
    raise ValueError("Figure 4 simulations contain missing/nonfinite values or SDs.")
if (figure4_sim["sd"] < 0).any():
    raise ValueError("Standard deviations must be nonnegative.")
# The CSV records the seed count and convergence status for each method.
# Plot unsquared cosine with the supplied sample standard deviation.
assert np.allclose(
    figure4_sim["sem"],
    figure4_sim["sd"] / np.sqrt(figure4_sim["num_seeds"]),
)

figure4_panels = (
    ("homogeneous", "(a) Homogeneous noise"),
    ("bounded", "(b) Bounded noise"),
    ("unbounded", "(c) Unbounded noise"),
)
figure4_width, figure4_height = 7.0, 1.8
figure4_title_fontsize = 9 * FONT_SCALE
figure4_label_fontsize = 8 * FONT_SCALE
figure4_tick_fontsize = 7 * FONT_SCALE
figure4_legend_fontsize = 6.3 * FONT_SCALE
figure4_linewidth = 1.3
scaling_factor = 1.3
# Draw wider curves first so narrower overlapping curves remain distinguishable.
figure4_theory_linewidths = {
    "bayes": figure4_linewidth*scaling_factor*scaling_factor,
    "dynamic": figure4_linewidth*scaling_factor*1.1,
    "static": figure4_linewidth*scaling_factor,
    "full_reconstruction": figure4_linewidth,
}
figure4_markersize = 3.8 * MARKER_SCALE
figure4_capsize = 2.0
figure4_sim_styles = {
    "amp": (BAYES_OPTIMAL_COLOR, "D", "AMP"),
    "dynamic": (DYNAMIC_MASKING_COLOR, "o", "Dynamic"),
    "static": (STATIC_MASKING_COLOR, "o", "Static"),
    "pca": (NO_MASKING_COLOR, "s", "PCA"),
    "full": (NO_MASKING_COLOR, "^", "Unmasked reconstruction"),
}
figure4_theory_styles = {
    "bayes": (BAYES_OPTIMAL_COLOR, "-", "Bayes-optimal"),
    "dynamic": (DYNAMIC_MASKING_COLOR, "-", "Dynamic"),
    "static": (STATIC_MASKING_COLOR, "-", "Static"),
    "full_reconstruction": (NO_MASKING_COLOR, "-", "Unmasked reconstruction"),
}
fig4, axes4 = plt.subplots(
    1, 3, figsize=(figure4_width, figure4_height),
    sharex=False, sharey=True, constrained_layout=True,
)
for ax, (panel, title) in zip(axes4, figure4_panels):
    panel_data = figure4_data.loc[figure4_data["panel"].eq(panel)]
    if panel_data.empty:
        raise ValueError(f"Missing noise setting: {panel}")
    for method, (color, linestyle, label) in figure4_theory_styles.items():
        subset = panel_data.loc[
            panel_data["kind"].eq("theory") & panel_data["method"].eq(method)
        ].sort_values("alpha")
        if subset.empty:
            continue
        # The asymptotic PCA result supplies this zero reference.
        # The archived failed solver rows are not treated as converged estimates.
        if panel == "unbounded" and method == "full_reconstruction":
            plotted_cosine = np.zeros(len(subset))
        else:
            valid = (
                ~subset["zero_imputed"]
                & ~subset["status"].fillna("").str.startswith("failed")
                & np.isfinite(subset["cosine"])
            )
            if not valid.any():
                continue
            plotted_cosine = subset["cosine"].where(valid)
        ax.plot(
            subset["alpha"], plotted_cosine,
            color=color, linestyle=linestyle, linewidth=figure4_theory_linewidths[method],
            zorder=2,
            label=label,
        )

    for method, (color, marker, label) in figure4_sim_styles.items():
        subset = panel_data.loc[
            panel_data["kind"].eq("simulation") & panel_data["method"].eq(method)
        ].sort_values("alpha")
        if subset.empty:
            continue  # Full-reconstruction simulations exist only for bounded noise.
        ax.errorbar(
            subset["alpha"], subset["cosine"], yerr=subset["sd"],
            fmt="none", ecolor=color, elinewidth=0.8,
            capsize=figure4_capsize, capthick=0.8, zorder=3, alpha=0.8,
        )
        ax.scatter(
            subset["alpha"], subset["cosine"], color=color, marker=marker,
            s=figure4_markersize ** 2, linewidths=0.3, zorder=4, alpha=0.8,
            label="_nolegend_",
        )

    ax.set_xscale("log")
    point_alpha = panel_data.loc[panel_data["kind"].eq("simulation"), "alpha"]
    point_alpha = point_alpha[np.isfinite(point_alpha) & (point_alpha > 0)]
    if point_alpha.empty:
        raise ValueError(f"No valid simulation alpha range for {panel}.")
    # Small multiplicative margin keeps endpoint markers visible on the log axis.
    alpha_min, alpha_max = float(point_alpha.min()) / 1.06, float(point_alpha.max()) * 1.06
    ax.set_xticks([
        tick for tick in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
        if alpha_min <= tick <= alpha_max
    ])
    ax.set_xlim(alpha_min, alpha_max)
    ax.set_ylim(-0.02, 1.0)
    from matplotlib.ticker import FixedLocator
    ax.yaxis.set_major_locator(FixedLocator([-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0]))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{value:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_title(title, fontsize=figure4_title_fontsize)
    ax.set_xlabel(r"Sample complexity $\alpha=n/d$", fontsize=figure4_label_fontsize)
    ax.tick_params(axis="both", labelsize=figure4_tick_fontsize)

fig4.supylabel(
    r"Cosine similarity, $\theta_{u^\star}$", fontsize=figure4_label_fontsize,
)
# Only state-evolution lines appear in the legend; points and SD bars stay plotted.
axes4[1].legend(
    loc="upper left", ncol=1,
    fontsize=figure4_legend_fontsize, frameon=True, framealpha=0.7,
    borderpad=0.3, labelspacing=0.25, handlelength=1.6, handletextpad=0.4,
)
figure4_omitted = figure4_data.loc[
    figure4_data["kind"].eq("theory") & figure4_data["zero_imputed"]
    & ~(
        figure4_data["panel"].eq("unbounded")
        & figure4_data["method"].eq("full_reconstruction")
    )
]
print(
    "Three-law recovery: cosine mean +/- SD across seeds; beta=1, d=2000, "
    "rho=0.75 for static/dynamic masking."
)
print("Unbounded PCA: analytical zero-alignment reference.")
print(
    "Excluded remaining imputed theory placeholders:",
    figure4_omitted.groupby(["panel", "method"]).size().to_dict(),
)
save_pdf(fig4, 'figure_02_noise_laws.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 3: Recovery and downstream accuracy across activations and mask diversity

Bounded Beta noise; beta=1, rho=0.75, ridge 1e-4. ERM uses d=2000 and 30 seeds per point. AMP uses d=4000, ten seeds and teacher initialization p=0.05. Upper panels show absolute cosine; lower panels use the same raw-feature logistic probe. Saved numerical statuses and residuals are retained.

Output: `plots/figure_03_activation_and_mask_diversity.pdf`.


In [ ]:
# New state evolution, 30-seed ERM, and matching 10-seed teacher-initialized AMP.
# Error bars are SD of the per-seed observables.
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

figure5_se_dir = DATA_ROOT / 'synthetic/bounded_alpha'
figure5_empirical_dir = figure5_se_dir
figure5_theory = pd.read_csv(figure5_se_dir / "state_evolution.csv")
figure5_theory["cosine"] = figure5_theory["cosine"].abs()  # Match sign-invariant empirical absolute cosine.
figure5_erm = pd.read_csv(figure5_empirical_dir / "erm_summary.csv")
figure5_bayes = pd.read_csv(figure5_se_dir / "bayes_optimal.csv")
figure5_amp = pd.read_csv(figure5_empirical_dir / "amp_d4000_oracle_plant0p05_summary.csv").sort_values("alpha")
# AMP is an activation-independent reference, repeated across all four columns.
for frame in (figure5_erm, figure5_amp):
    if "num_seeds" not in frame or not np.isfinite(frame["num_seeds"]).all() or (frame["num_seeds"] < 2).any():
        raise ValueError("SD requires a valid seed count of at least two per row.")
# Use probe test accuracy (the same observable as ERM), not Bayesian readout.
figure5_required_amp = {
    "alpha", "cosine_abs_mean", "cosine_abs_std",
    "probe_test_accuracy_mean", "probe_test_accuracy_std",
}
if not figure5_required_amp.issubset(figure5_amp.columns):
    raise ValueError("AMP summary is missing required mean/SD columns.")
if figure5_amp.empty or figure5_amp["alpha"].duplicated().any():
    raise ValueError("AMP summary must contain one row per alpha.")
if not np.isfinite(figure5_amp[sorted(figure5_required_amp)].to_numpy(float)).all():
    raise ValueError("AMP means and SDs must be finite.")
if (figure5_amp[["cosine_abs_std", "probe_test_accuracy_std"]] < 0).any().any():
    raise ValueError("AMP standard deviations must be nonnegative.")

figure5_required_theory = {
    "activation", "masking_protocol", "k_views", "alpha",
    "cosine", "downstream_test_accuracy",
}
figure5_required_erm = {
    "activation", "masking_protocol", "k_views", "alpha",
    "cosine_abs_mean", "cosine_abs_std",
    "probe_test_accuracy_mean", "probe_test_accuracy_std",
}
figure5_required_bayes = {
    "alpha", "bayes_optimal_cosine",
    "bayes_downstream_test_accuracy",
}
for frame, required, label in (
    (figure5_theory, figure5_required_theory, "state_evolution.csv"),
    (figure5_erm, figure5_required_erm, "erm_summary.csv"),
    (figure5_bayes, figure5_required_bayes, "bayes_optimal.csv"),
):
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{label} is missing columns: {sorted(missing)}")

figure5_activations = ("linear", "relu", "elu", "tanh")
figure5_activation_titles = {
    "linear": "Linear",
    "relu": "ReLU",
    "elu": "ELU",
    "tanh": "Tanh",
}
figure5_conditions = (
    ("finite_k_fixed", 1, r"$K=1$ (static)"),
    ("finite_k_fixed", 2, r"$K=2$"),
    ("finite_k_fixed", 4, r"$K=4$"),
    ("dynamic", 0, r"$K\rightarrow\infty$ (dynamic)"),
)
figure5_metrics = (
    ("cosine", "cosine_abs_mean", "cosine_abs_std",
     r"Cosine similarity, $\theta_{u^\star}$"),
    ("downstream_test_accuracy", "probe_test_accuracy_mean",
     "probe_test_accuracy_std", "Classification\naccuracy"),
)

# Double-column scale: readable font sizes are set directly in points.
figure5_width = 7.1
figure5_height = 3.8
figure5_title_fontsize = TWO_PANEL_STYLE["title"]
figure5_label_fontsize = TWO_PANEL_STYLE["label"]
figure5_tick_fontsize = TWO_PANEL_STYLE["tick"]
figure5_legend_fontsize = TWO_PANEL_STYLE["legend"]
figure5_linewidth = 1.35
figure5_markersize = 3.8 * MARKER_SCALE
figure5_capsize = 2.0
figure5_color_map = publication_k_colors(range(len(figure5_conditions)))
figure5_colors = tuple(
    figure5_color_map[index] for index in range(len(figure5_conditions))
)
figure5_markers = ("o",) * len(figure5_conditions)

fig5, axes5 = plt.subplots(
    2, 4,
    figsize=(figure5_width, figure5_height),
    sharex=True,
    sharey="row",
)

for column, activation in enumerate(figure5_activations):
    activation_theory = figure5_theory.loc[
        figure5_theory["activation"].eq(activation)
    ]
    activation_erm = figure5_erm.loc[
        figure5_erm["activation"].eq(activation)
    ]
    for row, (theory_col, mean_col, std_col, ylabel) in enumerate(
        figure5_metrics
    ):
        axis = axes5[row, column]
        
        for (protocol, k_views, _), color, marker in zip(
            figure5_conditions, figure5_colors, figure5_markers
        ):
            se = activation_theory.loc[
                activation_theory["masking_protocol"].eq(protocol)
                & activation_theory["k_views"].eq(k_views)
            ].sort_values("alpha")
            if se.empty:
                raise ValueError(
                    f"Missing state evolution for {activation}, {protocol}, K={k_views}"
                )
            # Every iterate is connected, independently of convergence status.
            axis.plot(
                se["alpha"], se[theory_col],
                color=color, linewidth=figure5_linewidth, zorder=2,
            )

            points = activation_erm.loc[
                activation_erm["masking_protocol"].eq(protocol)
                & activation_erm["k_views"].eq(k_views)
            ].sort_values("alpha")
            if points.empty:
                raise ValueError(
                    f"Missing ERM data for {activation}, {protocol}, K={k_views}"
                )
            axis.errorbar(
                points["alpha"], points[mean_col],
                yerr=points[std_col],
                color=color, marker=marker, linestyle="none",
                markersize=figure5_markersize,
                capsize=figure5_capsize, elinewidth=0.9, capthick=0.8,
                markerfacecolor=color, markeredgecolor=color, markeredgewidth=0,
                zorder=3,
            )
        

        bayes_column = (
            "bayes_optimal_cosine"
            if row == 0 else "bayes_downstream_test_accuracy"
        )
        axis.plot(
            figure5_bayes["alpha"], figure5_bayes[bayes_column],
            color=BAYES_OPTIMAL_COLOR, linestyle="-", linewidth=1.45, zorder=1,
        )
        axis.errorbar(
            figure5_amp["alpha"], figure5_amp[mean_col],
            yerr=figure5_amp[std_col],
            color=BAYES_OPTIMAL_COLOR, marker="D", linestyle="none",
            markersize=figure5_markersize,
            markerfacecolor=BAYES_OPTIMAL_COLOR, markeredgewidth=0,
            capsize=figure5_capsize, elinewidth=0.9, capthick=0.8,
            zorder=3,
        )
        axis.set_xscale("log")
        axis.set_xlim(0.1, 10.2)
        axis.set_xticks([0.1, 0.5, 1.0, 2.0, 5.0, 10.0])
        axis.set_xticklabels(["0.1", "0.5", "1", "2", "5", "10"])
        # Equal 4% padding beyond each row's outer ticks.
        axis.set_ylim((-0.04, 1.04) if row == 0 else (0.49, 0.76))
        axis.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0] if row == 0 else [0.5, 0.55, 0.6, 0.65, 0.7, 0.75])
        #axis.grid(True, linewidth=0.55, alpha=0.2)
        axis.tick_params(axis="both", which="major",
                         labelsize=figure5_tick_fontsize)
    
        
        if column == 0:
            axis.set_ylabel(ylabel, fontsize=figure5_label_fontsize)
        if row == 0:
            axis.set_title(
                figure5_activation_titles[activation],
                fontsize=figure5_title_fontsize,
            )

figure5_protocol_handles = [
    Line2D([0], [0], color=color, linewidth=1.5, label=label)
    for (_, _, label), color in zip(figure5_conditions, figure5_colors)
]
figure5_style_handles = [
    Line2D([0], [0], color=BAYES_OPTIMAL_COLOR, linewidth=figure5_linewidth, linestyle="-",
           label="Bayes-optimal"),
]
fig5.legend(
    handles=figure5_protocol_handles + figure5_style_handles,
    loc="lower center", bbox_to_anchor=(0.5, 0.12), ncol=6,
    fontsize=figure5_legend_fontsize, frameon=False,
    handlelength=2.0, columnspacing=1.2,
)
fig5.supxlabel(
    r"Sample complexity $\alpha=n/d$",
    fontsize=figure5_label_fontsize, y=0.2,
)
fig5.tight_layout(rect=(0, 0.14, 1, 0.98), pad=0.8, w_pad=0.5, h_pad=0.5)
# Keep the wrapped classification label aligned and clear of the page edge.
fig5.align_ylabels(axes5[:, 0])

figure5_output = PLOT_DIR / 'figure_03_activation_and_mask_diversity.pdf'


save_pdf(fig5, 'figure_03_activation_and_mask_diversity.pdf', dpi=300, bbox_inches='tight', pad_inches=0.12)

plt.show()


## Figure 4: Finite mask diversity in the synthetic model and CNN

Mask diversity is varied at fixed underlying data. The synthetic PCA reference is analytical. The original CNN sweep uses validation-selected encoders; see the separate fixed-budget control below.

Output: `plots/figure_04_finite_mask_diversity.pdf`.


In [ ]:
# Uncertainty: plotted error bars and shaded bands are sample standard deviation (SD).
# ============================================================
# ============================================================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


figure4_v3_spiked_data_dir = DATA_ROOT / 'synthetic/masking_ratio'
figure4_v3_theory = pd.read_csv(
    figure4_v3_spiked_data_dir / "generic_k_downstream_theory.csv"
)
figure4_v3_spiked = pd.read_csv(
    figure4_v3_spiked_data_dir / "k_view_downstream_summary.csv"
)
figure4_v3_cnn = pd.read_csv(
    DATA_ROOT / 'vision/cnn/mask_diversity/aggregate_test_accuracy.csv'
)
figure4_v3_endpoint_path = figure4_v3_spiked_data_dir / "downstream_probe_per_seed.csv"
if not figure4_v3_endpoint_path.is_file():
    raise FileNotFoundError(figure4_v3_endpoint_path)
figure4_v3_endpoints_raw = pd.read_csv(figure4_v3_endpoint_path)

figure4_v3_required_theory = {
    "mask_protocol", "mask_count", "rho", "status",
    "downstream_test_accuracy",
}
figure4_v3_required_spiked = {
    "k_views", "rho", "probe_test_accuracy_mean",
    "probe_test_accuracy_std",
}
figure4_v3_required_endpoints = {
    "activation", "method", "rho", "seed", "probe_test_accuracy",
}
figure4_v3_required_cnn = {
    "downstream_stage", "family", "rho",
    "n_independent_pretraining_and_split_seeds",
    "test_accuracy_mean", "test_accuracy_std",
}
for frame, required, source in (
    (
        figure4_v3_theory,
        figure4_v3_required_theory,
        "generic_k_downstream_theory.csv",
    ),
    (
        figure4_v3_spiked,
        figure4_v3_required_spiked,
        "k_view_downstream_summary.csv",
    ),
    (
        figure4_v3_endpoints_raw,
        figure4_v3_required_endpoints,
        figure4_v3_endpoint_path,
    ),
    (
        figure4_v3_cnn,
        figure4_v3_required_cnn,
        "aggregate_test_accuracy.csv",
    ),
):
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{source} is missing columns: {sorted(missing)}")

# Use the downstream linear-probing results for the CNN autoencoder.
figure4_v3_cnn = figure4_v3_cnn.loc[
    figure4_v3_cnn["downstream_stage"].eq("linear_probe")
].copy()
if figure4_v3_cnn.empty:
    raise ValueError(
        "aggregate_test_accuracy.csv has no linear_probe downstream rows."
    )
if figure4_v3_cnn.duplicated(["family", "rho"]).any():
    raise ValueError(
        "Expected one aggregate_test_accuracy.csv row per family and rho."
    )

figure4_v3_k_order = ("1", "2", "4", "8", "16", "32", "dynamic")
figure4_v3_labels = {
    "1": r"$K=1$ (static)",
    "2": r"$K=2$",
    "4": r"$K=4$",
    "8": r"$K=8$",
    "16": r"$K=16$",
    "32": r"$K=32$",
    "dynamic": r"$K\rightarrow\infty$ (dynamic)",
}
figure4_v3_theory_mask_counts = {
    "1": 1.0,
    "2": 2.0,
    "4": 4.0,
    "8": 8.0,
}
figure4_v3_endpoint_methods = {
    "1": "static_mask_mae_adam",
    "dynamic": "dynamic_mask_mae_adam",
}
figure4_v3_endpoints = (
    figure4_v3_endpoints_raw.loc[
        figure4_v3_endpoints_raw["activation"].eq("linear")
        & figure4_v3_endpoints_raw["method"].isin(
            figure4_v3_endpoint_methods.values()
        )
    ]
    .groupby(["method", "rho"], as_index=False)
    .agg(
        num_seeds=("seed", "nunique"),
        probe_test_accuracy_mean=("probe_test_accuracy", "mean"),
        probe_test_accuracy_std=("probe_test_accuracy", "std"),
    )
)
figure4_v3_cnn_families = {
    "1": "K1",
    "2": "K2",
    "4": "K4",
    "8": "K8",
    "16": "K16",
    "32": "K32",
    "dynamic": "dynamic_stepmatched",
}

figure4_v3_colors = publication_k_colors(figure4_v3_k_order)
figure4_v3_width, figure4_v3_height = TWO_PANEL_STYLE["figsize"]
figure4_v3_title_fontsize = TWO_PANEL_STYLE["title"]
figure4_v3_label_fontsize = TWO_PANEL_STYLE["label"]
figure4_v3_tick_fontsize = TWO_PANEL_STYLE["tick"]
figure4_v3_legend_fontsize = TWO_PANEL_STYLE["legend"]
figure4_v3_linewidth = 1.25
figure4_v3_markersize = 3.8 * MARKER_SCALE
figure4_v3_capsize = 2.0

fig4v3, axes4v3 = plt.subplots(
    1,
    2,
    figsize=(figure4_v3_width, figure4_v3_height),
    sharex=True,
    sharey=False,
    constrained_layout=True,
)

figure4_v3_top_bounds = []
for key in figure4_v3_k_order:
    if key in ("16", "32"):
        # K=16 and K=32 are available only for the CNN panel.
        axes4v3[0].plot(
            [],
            [],
            color=figure4_v3_colors[key],
            linewidth=figure4_v3_linewidth,
            label=figure4_v3_labels[key],
        )
        continue
    if key == "dynamic":
        theory_mask = figure4_v3_theory["mask_protocol"].eq(
            "annealed/dynamic"
        )
    else:
        theory_mask = (
            figure4_v3_theory["mask_protocol"].eq(
                "finite quenched masks"
            )
            & figure4_v3_theory["mask_count"].eq(
                figure4_v3_theory_mask_counts[key]
            )
        )
    theory = figure4_v3_theory.loc[
        theory_mask & figure4_v3_theory["status"].eq("converged")
    ].sort_values("rho")
    if theory.empty:
        raise ValueError(
            f"No converged theory rows found for {figure4_v3_labels[key]}."
        )
    rho = theory["rho"].to_numpy(float)
    accuracy = theory["downstream_test_accuracy"].to_numpy(float)
    valid = np.isfinite(rho) & np.isfinite(accuracy)
    if not valid.any():
        raise ValueError(
            f"No converged theory accuracy for {figure4_v3_labels[key]}."
        )
    figure4_v3_top_bounds.extend(accuracy[valid].tolist())
    axes4v3[0].plot(
        rho[valid],
        accuracy[valid],
        color=figure4_v3_colors[key],
        linewidth=figure4_v3_linewidth,
        zorder=2,
        label=figure4_v3_labels[key],
    )

# K=2,4,8 downstream points and sample standard deviations.
for k_views in sorted(
    figure4_v3_spiked["k_views"].dropna().unique()
):
    key = str(int(k_views))
    if key not in figure4_v3_colors:
        raise ValueError(f"Unexpected k_views={k_views!r}.")
    subset = figure4_v3_spiked.loc[
        figure4_v3_spiked["k_views"].eq(k_views)
    ].sort_values("rho")
    values = subset["probe_test_accuracy_mean"].to_numpy(float)
    std = subset["probe_test_accuracy_std"].to_numpy(float)
    if not np.isfinite(std).all():
        raise ValueError(
            f"Missing shallow-autoencoder standard deviations for K={key}."
        )
    figure4_v3_top_bounds.extend((values - std).tolist())
    figure4_v3_top_bounds.extend((values + std).tolist())
    axes4v3[0].errorbar(
        subset["rho"],
        values,
        yerr=std,
        fmt="none",
        ecolor=figure4_v3_colors[key],
        capsize=figure4_v3_capsize,
        zorder=3,
    )
    axes4v3[0].scatter(
        subset["rho"],
        values,
        color=figure4_v3_colors[key],
        marker="o",
        s=figure4_v3_markersize ** 2,
        zorder=4,
    )

# K=1 and dynamic downstream points are aggregated across linear seeds.
for key in ("1", "dynamic"):
    subset = figure4_v3_endpoints.loc[
        figure4_v3_endpoints["method"].eq(
            figure4_v3_endpoint_methods[key]
        )
    ].sort_values("rho")
    if subset.empty:
        raise ValueError(
            f"No empirical endpoint rows found for {figure4_v3_labels[key]}."
        )
    values = subset["probe_test_accuracy_mean"].to_numpy(float)
    std = subset["probe_test_accuracy_std"].to_numpy(float)
    if not np.isfinite(std).all():
        raise ValueError(
            f"Missing endpoint standard deviations for {figure4_v3_labels[key]}."
        )
    figure4_v3_top_bounds.extend((values - std).tolist())
    figure4_v3_top_bounds.extend((values + std).tolist())
    axes4v3[0].errorbar(
        subset["rho"],
        values,
        yerr=std,
        fmt="none",
        ecolor=figure4_v3_colors[key],
        capsize=figure4_v3_capsize,
        zorder=3,
    )
    axes4v3[0].scatter(
        subset["rho"],
        values,
        color=figure4_v3_colors[key],
        marker="o",
        s=figure4_v3_markersize ** 2,
        zorder=4,
    )

# Asymptotic PCA for the same bounded law: beta*m_+ = 8/9 < 1.
# This theoretical chance reference has no empirical uncertainty band.
figure4_v3_pca_asymptotic_accuracy = 0.5
figure4_v3_top_bounds.append(figure4_v3_pca_asymptotic_accuracy)
# The shared Unmasked reconstruction legend entry is supplied by the CNN panel.
axes4v3[0].axhline(
    figure4_v3_pca_asymptotic_accuracy,
    color=NO_MASKING_COLOR, linewidth=1.38,
    alpha=1.0, zorder=2, label="_nolegend_",
)

figure4_v3_bottom_bounds = []
for key in figure4_v3_k_order:
    subset = figure4_v3_cnn.loc[
        figure4_v3_cnn["family"].eq(figure4_v3_cnn_families[key])
    ].sort_values("rho")
    if subset.empty:
        raise ValueError(
            f"No CNN rows found for {figure4_v3_labels[key]}."
        )
    values = subset["test_accuracy_mean"].to_numpy(float)
    std = subset["test_accuracy_std"].to_numpy(float)
    if not np.isfinite(std).all():
        raise ValueError(
            f"Missing CNN standard deviations for {figure4_v3_labels[key]}."
        )
    figure4_v3_bottom_bounds.extend((values - std).tolist())
    figure4_v3_bottom_bounds.extend((values + std).tolist())
    axes4v3[1].errorbar(
        subset["rho"],
        values,
        yerr=std,
        fmt="none",
        ecolor=figure4_v3_colors[key],
        capsize=figure4_v3_capsize,
        zorder=3,
    )
    axes4v3[1].scatter(
        subset["rho"],
        values,
        color=figure4_v3_colors[key],
        marker="o",
        s=figure4_v3_markersize ** 2,
        zorder=4,
    )


# CNN no-masking baseline: the same linear-probe snapshot used in Figure 1.
figure4_v3_no_mask_path = (
    DATA_ROOT / 'vision/cnn/masking_ratio/comparison_plot_data.csv'
)
figure4_v3_no_mask_data = pd.read_csv(figure4_v3_no_mask_path)
figure4_v3_no_mask_row = figure4_v3_no_mask_data.loc[
    figure4_v3_no_mask_data["method"].eq("Full reconstruction")
]
if len(figure4_v3_no_mask_row) != 1:
    raise ValueError("Expected one CNN full-reconstruction baseline.")
figure4_v3_no_mask_mean = float(figure4_v3_no_mask_row.iloc[0]["test_accuracy_mean"])
figure4_v3_no_mask_std = float(figure4_v3_no_mask_row.iloc[0]["test_accuracy_std"])
figure4_v3_no_mask_sd = figure4_v3_no_mask_std
if (
    not np.isfinite([figure4_v3_no_mask_mean, figure4_v3_no_mask_std]).all()
    or figure4_v3_no_mask_std < 0
    or int(figure4_v3_no_mask_row.iloc[0]["n_independent_pretraining_and_split_seeds"]) < 2
):
    raise ValueError("CNN no-masking baseline requires a finite mean and sample SD.")
figure4_v3_bottom_bounds.extend([
    figure4_v3_no_mask_mean - figure4_v3_no_mask_sd,
    figure4_v3_no_mask_mean + figure4_v3_no_mask_sd,
])
axes4v3[1].axhspan(
    figure4_v3_no_mask_mean - figure4_v3_no_mask_sd,
    figure4_v3_no_mask_mean + figure4_v3_no_mask_sd,
    color=NO_MASKING_COLOR, alpha=0.6, linewidth=0, zorder=1,
)
figure4_v3_no_mask_line = axes4v3[1].axhline(
    figure4_v3_no_mask_mean,
    color=NO_MASKING_COLOR, linewidth=1.38, alpha=1.0, zorder=2,
    label="Unmasked reconstruction",
)
axes4v3[0].set_title(
    publication_title("(a) Shallow Autoencoder", "Spiked model"),
    fontsize=figure4_v3_title_fontsize,
)
axes4v3[1].set_title(
    publication_title("(b) CNN Autoencoder", "CIFAR-10"),
    fontsize=figure4_v3_title_fontsize,
)
axes4v3[0].set_ylabel(
        "Classification accuracy",
        fontsize=figure4_v3_label_fontsize,
    )
for ax in axes4v3:
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel(
        r"Masking ratio $\rho$",
        fontsize=figure4_v3_label_fontsize,
    )

    ax.tick_params(
        axis="both",
        labelsize=figure4_v3_tick_fontsize,
    )

figure4_v3_legend_handles, figure4_v3_legend_labels = (
    axes4v3[0].get_legend_handles_labels()
)
fig4v3.legend(
    handles=figure4_v3_legend_handles + [figure4_v3_no_mask_line],
    labels=figure4_v3_legend_labels + ["Unmasked reconstruction"],
    loc="upper center", bbox_to_anchor=(0.5, 0.03),
    ncol=8,
    fontsize=figure4_v3_legend_fontsize,
    frameon=False,
    handlelength=1.3,
    handletextpad=0.35,
    columnspacing=0.8,
)


# ============================================================
# EDIT Y AXES HERE (ticks first, limits last to keep the chosen range)
# ============================================================
# Left: shallow autoencoder
axes4v3[0].set_yticks([0.50, 0.55, 0.60, 0.65, 0.70, 0.75])
axes4v3[0].set_ylim(0.48, 0.7765884130829716)

# Right: CNN autoencoder
axes4v3[1].set_yticks([0.40, 0.45, 0.50, 0.55, 0.60])
axes4v3[1].set_ylim(0.395, 0.63)

save_pdf(plt.gcf(), 'figure_04_finite_mask_diversity.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 5: ViT augmentation comparison

One paired pretraining seed for each condition. All masking ratios are included, with and without random cropping/flipping. No across-seed uncertainty is estimated.

Output: `plots/figure_05_vit_augmentation.pdf`.


In [ ]:
# ViT augmentation comparison: seed 0 only, no error bars.
# Both conditions come from one CSV; seed identity supplied with the dataset.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

vit_aug_dir = PLOT_DIR
vit_aug_data = pd.read_csv(DATA_ROOT / 'vision/vit/augmentation/linear_probe_val_accuracy_geom_vs_no_geom.csv')
if set(vit_aug_data["family"]) != {"lambda_0", "no_geom_lambda_0"}:
    raise ValueError("Unexpected augmentation families.")
if vit_aug_data.duplicated(["family", "mode", "rho"]).any():
    raise ValueError("Duplicate seed-0 comparison rows.")
if not np.isfinite(vit_aug_data[["rho", "val_acc5"]]).all().all():
    raise ValueError("Missing masking ratios or top-5 accuracy.")
if not vit_aug_data["val_acc5"].between(0, 1).all():
    raise ValueError("Top-5 accuracy must be a fraction.")
vit_aug_no = vit_aug_data.loc[vit_aug_data["family"].eq("no_geom_lambda_0")].copy()
vit_aug_yes = vit_aug_data.loc[vit_aug_data["family"].eq("lambda_0")].copy()
if not set(zip(vit_aug_no["mode"], vit_aug_no["rho"])) == set(zip(vit_aug_yes["mode"], vit_aug_yes["rho"])):
    raise ValueError("Augmentation conditions have different method/rho grids.")

fig_vit_aug, ax_vit_aug = plt.subplots(
    figsize=(TWO_PANEL_STYLE["figsize"][0] / 2, TWO_PANEL_STYLE["figsize"][1]),
    constrained_layout=True,
)
vit_aug_styles = (
    ("dynamic", DYNAMIC_MASKING_COLOR, "Dynamic masking"),
    ("static", STATIC_MASKING_COLOR, "Static masking"),
)
for mode, color, label in vit_aug_styles:
    left = vit_aug_no.loc[vit_aug_no["mode"].eq(mode)].sort_values("rho")
    right = vit_aug_yes.loc[vit_aug_yes["mode"].eq(mode)].sort_values("rho")
    if left.empty or right.empty:
        raise ValueError(f"Missing ViT results for {mode}.")
    # Single seed: no across-seed SD can be estimated.
    ax_vit_aug.scatter(
        left["rho"], left["val_acc5"],
        facecolors="white", edgecolors=color, marker="o", s=(3.8 * MARKER_SCALE) ** 2, linewidths=1.4,
        zorder=4, label=label.replace(" masking", "") + ", no augmentation",
    )
    # The new CSV provides one run per point, so no SD can be estimated.
    ax_vit_aug.scatter(
        right["rho"], right["val_acc5"],
        color=color, marker="x", s=(3.8 * MARKER_SCALE) ** 2, linewidths=1.4,
        zorder=4, label=label.replace(" masking", "") + ", standard augmentation",
    )

ax_vit_aug.set_title(publication_title("ViT-Base", "ImageNet-100"), fontsize=TWO_PANEL_STYLE["title"])
ax_vit_aug.set_ylabel("Classification accuracy", fontsize=TWO_PANEL_STYLE["label"])
ax_vit_aug.set_xlabel(r"Masking ratio $\rho$", fontsize=TWO_PANEL_STYLE["label"])
ax_vit_aug.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax_vit_aug.set_xlim(0.0, 1.0)
ax_vit_aug.tick_params(axis="both", labelsize=TWO_PANEL_STYLE["tick"])
# Independent color and shape keys avoid repeating all four combinations.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
vit_aug_handles = [
    Patch(facecolor=DYNAMIC_MASKING_COLOR, edgecolor="none", label="Dynamic"),
    Patch(facecolor=STATIC_MASKING_COLOR, edgecolor="none", label="Static"),
    Line2D([], [], color="black", marker="o", linestyle="none",
           markersize=3.8 * MARKER_SCALE, markerfacecolor="white", markeredgewidth=1.4, label="Without cropping/flipping"),
    Line2D([], [], color="black", marker="x", linestyle="none",
           markersize=3.8 * MARKER_SCALE, markeredgewidth=1.4, label="With cropping/flipping"),
]
fig_vit_aug.legend(
    handles=vit_aug_handles, loc="upper center",
    bbox_to_anchor=(0.5, 0.03), ncol=2, fontsize=TWO_PANEL_STYLE["legend"],
    frameon=False, handlelength=1.3, handletextpad=0.35, columnspacing=0.8,
)

# EDIT Y AXIS HERE
ax_vit_aug.set_yticks([0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75])
#ax_vit_aug.set_ylim(0.45, 0.55)

vit_aug_output = PLOT_DIR / 'figure_05_vit_augmentation.pdf'
save_pdf(fig_vit_aug, 'figure_05_vit_augmentation.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 6: PCA dimension scaling

Absolute PCA cosine, 30 seeds per point. The right panel uses the three largest sample complexities, d>=1600, and amplitude-only fits with the exponent fixed at -1/2.

Output: `plots/figure_06_pca_dimension_scaling.pdf`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, NullFormatter

a2_dir = DATA_ROOT / 'spectral/dimension_scaling'
a2_summary = pd.read_csv(a2_dir / "pca_summary.csv")
a2_fits = pd.read_csv(a2_dir / "pca_cosine_dimension_fixed_slope_fits.csv")
a2_fits = a2_fits.loc[
    a2_fits["metric"].eq("cosine_abs") & a2_fits["fit_scope"].eq("large_dimension_tail")
].sort_values("alpha")
a2_dims = np.sort(a2_summary["d"].unique())
if a2_summary.duplicated(["d", "alpha"]).any() or len(a2_summary) != 90:
    raise ValueError("Expected the complete 9-dimension by 10-alpha summary.")
if not a2_summary["num_seeds"].eq(30).all():
    raise ValueError("Expected 30 seeds per point.")
if not np.isfinite(a2_summary[["cosine_abs_mean", "cosine_abs_std", "cosine_abs_sem"]]).all().all():
    raise ValueError("Nonfinite PCA summary values.")
if not np.allclose(a2_summary["cosine_abs_sem"],
                   a2_summary["cosine_abs_std"] / np.sqrt(a2_summary["num_seeds"])):
    raise ValueError("SEM does not match SD / sqrt(number of seeds).")
if len(a2_fits) != 3 or not np.allclose(a2_fits["p_fixed"], 0.5):
    raise ValueError("Expected three saved fixed-exponent fits.")

# Independent palettes for dimension and alpha (not the masking-K palette).
A2_DIMENSION_COLORMAP = "magma_r"
A2_DIMENSION_COLOR_MIN, A2_DIMENSION_COLOR_MAX = 0.10, 0.85
A2_ALPHA_COLORS = ("#009E73", "#0072B2", "#9467BD")
a2_cmap = LinearSegmentedColormap.from_list(
    "appendix_dimension_colors",
    plt.get_cmap(A2_DIMENSION_COLORMAP)(np.linspace(A2_DIMENSION_COLOR_MIN, A2_DIMENSION_COLOR_MAX, 256)),
)
a2_norm = LogNorm(vmin=a2_dims.min(), vmax=a2_dims.max())
a2_alpha_colors = dict(zip(a2_fits["alpha"].tolist(), A2_ALPHA_COLORS))
fig_a2, axes_a2 = plt.subplots(1, 2, figsize=TWO_PANEL_STYLE["figsize"], constrained_layout=True)

for dimension in a2_dims:
    rows = a2_summary.loc[a2_summary["d"].eq(dimension)].sort_values("alpha")
    if len(rows) != 10:
        raise ValueError(f"Incomplete alpha grid for d={dimension}.")
    axes_a2[0].errorbar(
        rows["alpha"], rows["cosine_abs_mean"], yerr=rows["cosine_abs_std"],
        fmt="o", linestyle="none", color=a2_cmap(a2_norm(dimension)),
        markersize=3.8 * MARKER_SCALE, markeredgewidth=0,
        capsize=2, capthick=0.8, elinewidth=0.9,
    )
axes_a2[0].set_xscale("log")
axes_a2[0].set_title("(a) PCA finite-size scaling", fontsize=TWO_PANEL_STYLE["title"])
axes_a2[0].set_xlabel(r"Sample complexity $\alpha=n/d$", fontsize=TWO_PANEL_STYLE["label"])
axes_a2[0].set_ylabel(r"Cosine similarity, $\theta_{u^\star}$", fontsize=TWO_PANEL_STYLE["label"])

for fit in a2_fits.itertuples(index=False):
    rows = a2_summary.loc[
        np.isclose(a2_summary["alpha"], fit.alpha)
        & a2_summary["d"].between(fit.d_min, fit.d_max)
    ].sort_values("d")
    if len(rows) != fit.n_dimensions:
        raise ValueError("Fit dimension range does not match the plotted points.")
    # Independently check the exported amplitude against the stated estimator.
    expected_amplitude = np.exp(np.mean(np.log(rows["cosine_abs_mean"]) + 0.5 * np.log(rows["d"])))
    if not np.isclose(expected_amplitude, fit.amplitude_A):
        raise ValueError("Saved amplitude does not match the source data.")
    color = a2_alpha_colors[fit.alpha]
    axes_a2[1].errorbar(
        rows["d"], rows["cosine_abs_mean"], yerr=rows["cosine_abs_std"],
        fmt="o", linestyle="none", color=color,
        markersize=3.8 * MARKER_SCALE, markeredgewidth=0,
        capsize=2, capthick=0.8, elinewidth=0.9,
    )
    d_fit = np.geomspace(fit.d_min, fit.d_max, 200)
    axes_a2[1].plot(d_fit, fit.amplitude_A * d_fit ** (-fit.p_fixed),
                    color=color, linestyle="--", linewidth=1.35)
axes_a2[1].set_xscale("log")
axes_a2[1].set_yscale("log")
axes_a2[1].set_title(r"(b) $d^{-1/2}$ scaling", fontsize=TWO_PANEL_STYLE["title"])
axes_a2[1].set_xlabel(r"Dimension $d$", fontsize=TWO_PANEL_STYLE["label"])
axes_a2[1].set_ylabel(r"Cosine similarity, $\theta_{u^\star}$", fontsize=TWO_PANEL_STYLE["label"])
for axis in axes_a2:
    axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])

a2_handles = [
    Line2D([], [], linestyle="none", marker="o", color=a2_alpha_colors[alpha],
           markersize=3.8 * MARKER_SCALE, markeredgewidth=0,
           label=rf"$\alpha={alpha:.2f}$")
    for alpha in a2_fits["alpha"]
]
a2_handles.append(Line2D([], [], color="black", linestyle="--", linewidth=1.35,
                         label=r"Fit: $A_\alpha d^{-1/2}$"))
axes_a2[1].legend(handles=a2_handles, loc="lower left",
               ncol=2, frameon=False, fontsize=TWO_PANEL_STYLE["legend"],
               handlelength=1.8, columnspacing=1.1)

# EDIT AXIS LIMITS AND TICKS HERE.
axes_a2[0].set_xlim(0.85, 60)
axes_a2[0].set_xticks([1, 2, 5, 10, 20, 50])
axes_a2[0].xaxis.set_major_formatter(ScalarFormatter())
axes_a2[0].set_ylim(0, 0.60)
axes_a2[0].set_yticks([0, 0.2, 0.4, 0.6])
axes_a2[1].set_xlim(1400, 30000)
axes_a2[1].set_xticks([1600, 6400, 25600])
axes_a2[1].xaxis.set_major_formatter(ScalarFormatter())
axes_a2[1].set_ylim(0.035, 0.4)
axes_a2[1].set_yticks([0.05, 0.1, 0.2, 0.4])
axes_a2[1].yaxis.set_major_formatter(ScalarFormatter())
axes_a2[1].yaxis.set_minor_formatter(NullFormatter())

a2_colorbar = fig_a2.colorbar(ScalarMappable(norm=a2_norm, cmap=a2_cmap),
                             ax=axes_a2[0], fraction=0.035, pad=0.025)
a2_colorbar.ax.set_title(r"$d$", fontsize=TWO_PANEL_STYLE["label"])
a2_colorbar.set_ticks([100, 1600, 25600])
a2_colorbar.set_ticklabels(["100", "1600", "25600"])
a2_colorbar.ax.tick_params(labelsize=TWO_PANEL_STYLE["tick"])
a2_colorbar.minorticks_off()

a2_output = PLOT_DIR / 'figure_06_pca_dimension_scaling.pdf'
save_pdf(fig_a2, 'figure_06_pca_dimension_scaling.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 7: ViT top-1 validation accuracy

Left: top-1 validation accuracy over three initialization seeds. Right: the single-seed augmentation comparison, without error bars.

Output: `plots/figure_07_vit_top1.pdf`.


In [ ]:
# ViT-Base top-1 validation accuracy: multi-seed results and augmentation comparison.
# Left: Figure 1's matching top-1 snapshot; sample SD is plotted directly.
# Right: seed 0 only, as supplied in the augmentation CSV; no error bars.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

if "TWO_PANEL_STYLE" not in globals():
    raise RuntimeError("Run the first cell to initialize the publication style.")
vit_top1_mean = pd.read_csv(DATA_ROOT / 'vision/vit/masking_ratio'
                           / "linear_probe_top1_accuracy_mean_across_initialization_seeds.csv")
vit_top1_aug = pd.read_csv(DATA_ROOT / 'vision/vit/augmentation/linear_probe_val_accuracy_geom_vs_no_geom.csv')
vit_top1_curves = vit_top1_mean.loc[vit_top1_mean["plot_element"].eq("masking_curve")].copy()
if (len(vit_top1_curves) != 16
    or vit_top1_curves.duplicated(["masking_mode", "rho"]).any()
    or not vit_top1_curves["number_of_seeds"].eq(3).all()
    or not vit_top1_curves["augmentation"].eq("no_geom").all()):
    raise ValueError("Expected 16 unaugmented masking groups with three seeds each.")
vit_top1_seed_values = vit_top1_curves[
    ["seed_0_top1_accuracy", "seed_1_top1_accuracy", "seed_2_top1_accuracy"]].to_numpy(float)
if (not np.isfinite(vit_top1_seed_values).all()
    or not np.allclose(vit_top1_seed_values.mean(axis=1), vit_top1_curves["mean_top1_accuracy"])
    or not np.allclose(vit_top1_seed_values.std(axis=1, ddof=1),
                       vit_top1_curves["sample_std_top1_accuracy"])):
    raise ValueError("Top-1 means/sample SD disagree with the per-seed values.")
vit_top1_curves["sd"] = (vit_top1_curves["sample_std_top1_accuracy"])
if (len(vit_top1_aug) != 32
    or vit_top1_aug.duplicated(["family", "mode", "rho"]).any()
    or set(vit_top1_aug["family"]) != {"lambda_0", "no_geom_lambda_0"}
    or set(vit_top1_aug["mode"]) != {"dynamic", "static"}):
    raise ValueError("Expected 32 unique seed-0 augmentation comparison rows.")
if (not np.isfinite(vit_top1_aug[["rho", "val_acc1"]]).all().all()
    or not vit_top1_aug["val_acc1"].between(0, 1).all()
    or not np.allclose(100 * vit_top1_aug["val_acc1"], vit_top1_aug["val_acc1_percent"])):
    raise ValueError("Invalid top-1 validation accuracy or inconsistent percent units.")

fig_vit_top1, axes_vit_top1 = plt.subplots(
    1, 2, figsize=TWO_PANEL_STYLE["figsize"], sharey=True, constrained_layout=True)
for mode, color, label in (("dynamic", DYNAMIC_MASKING_COLOR, "Dynamic"),
                            ("static", STATIC_MASKING_COLOR, "Static")):
    rows = vit_top1_curves.loc[vit_top1_curves["masking_mode"].eq(mode)].sort_values("rho")
    if len(rows) != 8:
        raise ValueError(f"Expected eight masking ratios for {mode}.")
    axes_vit_top1[0].errorbar(
        rows["rho"], rows["mean_top1_accuracy"], yerr=rows["sd"],
        fmt="o", linestyle="none", color=color, markerfacecolor=color,
        markersize=3.8 * MARKER_SCALE, markeredgewidth=0.8,
        capsize=2.0, elinewidth=0.9, capthick=0.8, zorder=4, label=label)
    for family, marker in (("no_geom_lambda_0", "o"), ("lambda_0", "x")):
        points = vit_top1_aug.loc[vit_top1_aug["family"].eq(family)
                                  & vit_top1_aug["mode"].eq(mode)].sort_values("rho")
        if len(points) != 8 or not np.allclose(points["rho"], rows["rho"]):
            raise ValueError("The two panels must have matching masking-ratio grids.")
        if family == "no_geom_lambda_0" and not np.allclose(
                points["val_acc1"], rows["seed_0_top1_accuracy"]):
            raise ValueError("Augmentation snapshot disagrees with the unaugmented seed-0 values.")
        marker_style = ({"facecolors": "white", "edgecolors": color}
                        if marker == "o" else {"color": color})
        axes_vit_top1[1].scatter(
            points["rho"], points["val_acc1"], marker=marker,
            s=(3.8 * MARKER_SCALE)**2, linewidths=1.4, zorder=4, **marker_style)

for element, color, label in (
    ("horizontal_reference", NO_MASKING_COLOR, "Unmasked reconstruction"),
    ("standard_geometry_horizontal_reference", STANDARD_AUGMENTATION_COLOR,
     "Cropping/flipping + dynamic masking"),
):
    reference = vit_top1_mean.loc[vit_top1_mean["plot_element"].eq(element)]
    if (reference.empty or not reference["number_of_seeds"].eq(1).all()
        or not np.isfinite(reference["mean_top1_accuracy"]).all()
        or np.ptp(reference["mean_top1_accuracy"].to_numpy(float)) > 1e-10):
        raise ValueError(f"Expected a constant single-seed reference for {element}.")
    # Repeated baseline rows are one run, not independent seeds; no SD band.
    axes_vit_top1[0].axhline(float(reference.iloc[0]["mean_top1_accuracy"]),
                             color=color, linewidth=1.38, zorder=2, label=label)

for axis, title in zip(axes_vit_top1,
                      ("(a) ViT-Base: three-seed comparison",
                       "(b) ViT-Base: augmentation comparison")):
    axis.set_title(publication_title(title, "ImageNet-100"),
                   fontsize=TWO_PANEL_STYLE["title"])
    axis.set_xlabel(r"Masking ratio $\rho$", fontsize=TWO_PANEL_STYLE["label"])
    axis.set_xlim(0, 1)
    axis.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axis.tick_params(axis="both", labelsize=TWO_PANEL_STYLE["tick"], labelleft=True)
axes_vit_top1[0].set_ylabel("Top-1 validation accuracy", fontsize=TWO_PANEL_STYLE["label"])

# Legends below their respective panels, using the shared publication typography.
vit_top1_handles, vit_top1_labels = axes_vit_top1[0].get_legend_handles_labels()
vit_top1_by_label = dict(zip(vit_top1_labels, vit_top1_handles))
vit_top1_order = ["Dynamic", "Static", "Unmasked reconstruction", "Cropping/flipping + dynamic masking"]
fig_vit_top1.legend(
    [vit_top1_by_label[label] for label in vit_top1_order], vit_top1_order,
    loc="upper center", bbox_to_anchor=(0.27, 0.02), ncol=2,
    fontsize=TWO_PANEL_STYLE["legend"], frameon=False,
    handlelength=1.3, handletextpad=0.35, columnspacing=0.8)
vit_top1_aug_handles = [
    Patch(facecolor=DYNAMIC_MASKING_COLOR, edgecolor="none", label="Dynamic"),
    Patch(facecolor=STATIC_MASKING_COLOR, edgecolor="none", label="Static"),
    Line2D([], [], color="black", marker="o", linestyle="none", markerfacecolor="white",
           markersize=3.8 * MARKER_SCALE, markeredgewidth=1.4, label="Without cropping/flipping"),
    Line2D([], [], color="black", marker="x", linestyle="none",
           markersize=3.8 * MARKER_SCALE, markeredgewidth=1.4, label="With cropping/flipping"),
]
fig_vit_top1.legend(
    handles=vit_top1_aug_handles, loc="upper center", bbox_to_anchor=(0.77, 0.02), ncol=2,
    fontsize=TWO_PANEL_STYLE["legend"], frameon=False,
    handlelength=1.3, handletextpad=0.35, columnspacing=0.8)

# EDIT Y AXIS HERE: shared limits/ticks make the two panels directly comparable.
axes_vit_top1[0].set_ylim(0.14, 0.43)
axes_vit_top1[0].set_yticks([0.15, 0.20, 0.25, 0.30, 0.35, 0.40])

vit_top1_output = PLOT_DIR / 'figure_07_vit_top1.pdf'
save_pdf(fig_vit_top1, 'figure_07_vit_top1.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("Left: three-seed mean +/- SD; baseline lines each use one seed (no uncertainty band).")
print("Right: seed 0 only, no error bars. Both panels report frozen linear-probe top-1 validation accuracy.")


## Figure 8: Spectral estimators under heterogeneous noise

Bounded and shifted-lognormal noise, beta=1. PCA variants and dynamic masking use d=2000 and 20 seeds; AMP uses d=4000 and ten teacher-initialized seeds (p=0.05). Hollow AMP markers identify groups with iteration-limited runs. Only plot-eligible rows are drawn.

Output: `plots/figure_08_spectral_estimators.pdf`.


In [ ]:
# All plotting is inline; only the packaged comparison CSV is loaded.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter

a4_dir = DATA_ROOT / 'spectral/heterogeneous_estimators'
a4_data = pd.read_csv(a4_dir / "comparison.csv")
for flag in ("plot_eligible", "zero_imputed", "archived_nonconverged"):
    a4_data[flag] = a4_data[flag].astype(str).str.lower().isin(["true", "1", "1.0"])
if not a4_data["case_id"].eq("original_" + a4_data["panel"]).all() or not a4_data["beta"].eq(1).all():
    raise ValueError("Expected the original bounded/unbounded beta=1 settings.")
A4_ERROR = "sd"  # Sample standard deviation across seeds.
A4_ALTERNATIVE_PCA_MARKER_SCALE = 1.4
A4_ALTERNATIVE_PCA_METHODS = ("diagonal_deletion_pca", "heteropca", "estimated_whitening_pca")
def a4_marker_size(method):
    return 3.8 * MARKER_SCALE * (A4_ALTERNATIVE_PCA_MARKER_SCALE if method in A4_ALTERNATIVE_PCA_METHODS else 1.0)

a4_styles = {
    "bayes": (BAYES_OPTIMAL_COLOR, "", "Bayes-optimal"),
    "amp": (BAYES_OPTIMAL_COLOR, "D", "AMP"),
    "dynamic": (DYNAMIC_MASKING_COLOR, "o", "Dynamic"),
    "full_reconstruction": (NO_MASKING_COLOR, "", "Unmasked reconstruction"),
    "pca": (NO_MASKING_COLOR, "s", "PCA"),
    "diagonal_deletion_pca": ("#E66101", "v", "Diagonal-deletion PCA"),
    "heteropca": ("#9467BD", "^", "HeteroPCA"),
    "estimated_whitening_pca": ("#009E73", "P", "Estimated-whitening PCA"),
}
if set(a4_data["method"]) != set(a4_styles):
    raise ValueError("Unexpected methods in the comparison export.")
if a4_data.duplicated(["panel", "kind", "method", "alpha"]).any():
    raise ValueError("Duplicate comparison rows.")
a4_sims = a4_data.loc[a4_data["kind"].eq("simulation")]
if not np.isfinite(a4_sims[["cosine", "sd", "sem", "num_seeds"]]).all().all():
    raise ValueError("Missing simulation means or uncertainty.")
if (a4_sims[["sd", "sem"]] < 0).any().any() or (a4_sims["num_seeds"] < 2).any():
    raise ValueError("Invalid seed counts or uncertainties.")
if not np.allclose(a4_sims["sem"], a4_sims["sd"] / np.sqrt(a4_sims["num_seeds"])):
    raise ValueError("SEM does not match SD / sqrt(number of seeds).")
fig_a4, axes_a4 = plt.subplots(1, 2, figsize=TWO_PANEL_STYLE["figsize"],
                              sharey=True, constrained_layout=True)
a4_omitted = 0
for axis, panel, title in zip(
    axes_a4, ["bounded", "unbounded"], ["(a) Bounded noise", "(b) Unbounded noise"]
):
    for method, (color, marker, label) in a4_styles.items():
        rows = a4_data.loc[a4_data["panel"].eq(panel) & a4_data["method"].eq(method)]
        theory = rows.loc[rows["kind"].eq("theory")].sort_values("alpha")
        if not theory.empty:
            # Use the export eligibility flags; never display failed/imputed theory.
            valid = theory["plot_eligible"] & ~theory["zero_imputed"]
            values = theory["cosine"].where(valid)
            axis.plot(theory["alpha"], values, color=color,
                      linewidth=2.0 if method == "bayes" else (1.6 if method == "dynamic" else 1.2),
                      zorder=4 if method == "bayes" else 2)
        points = rows.loc[rows["kind"].eq("simulation")].sort_values("alpha")
        a4_omitted += int((~points["plot_eligible"]).sum())
        points = points.loc[points["plot_eligible"]]
        if method in A4_ALTERNATIVE_PCA_METHODS and not points["num_converged"].eq(points["num_seeds"]).all():
            raise ValueError("Eligible PCA results must have all seeds converged.")
        for capped, group in points.groupby("archived_nonconverged"):
            axis.errorbar(group["alpha"], group["cosine"], yerr=group[A4_ERROR],
                          fmt=marker, linestyle="none", color=color,
                          markersize=a4_marker_size(method),
                          markerfacecolor="none" if capped else color,
                          markeredgewidth=0.8 if capped else 0,
                          capsize=2, capthick=0.8, elinewidth=0.8, alpha=0.8,
                          zorder=7 if method == "estimated_whitening_pca" else (6 if method == "amp" else 5))
    axis.set_title(title, fontsize=TWO_PANEL_STYLE["title"])
    axis.set_xscale("log")
    axis.set_xlabel(r"Sample complexity $\alpha=n/d$", fontsize=TWO_PANEL_STYLE["label"])
    axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])
axes_a4[0].set_ylabel(r"Cosine similarity, $\theta_{u^\star}$", fontsize=TWO_PANEL_STYLE["label"])
# Match Figure 4: reference methods use line-only legend entries.
# AMP and ordinary PCA points remain plotted without separate legend entries.
a4_handles = [
    Line2D([], [], color=a4_styles[method][0], linewidth=1.35,
           label=a4_styles[method][2])
    for method in ("bayes", "dynamic", "full_reconstruction")
] + [
    Line2D([], [], color=a4_styles[method][0], marker=a4_styles[method][1],
           markersize=a4_marker_size(method), markeredgewidth=0,
           linestyle="none", label=a4_styles[method][2])
    for method in A4_ALTERNATIVE_PCA_METHODS
]

fig_a4.legend(handles=a4_handles, loc="upper center", bbox_to_anchor=(0.5, 0.02),
               ncol=3, frameon=False, fontsize=TWO_PANEL_STYLE["legend"],
               handlelength=1.6, columnspacing=1.0)

# EDIT AXIS LIMITS AND TICKS HERE.
axes_a4[0].set_xlim(0.09, 11)
axes_a4[1].set_xlim(0.45, 11)
axes_a4[0].set_xticks([0.1, 0.5, 1, 2, 5, 10])
axes_a4[1].set_xticks([0.5, 1, 2, 5, 10])
for axis in axes_a4:
    axis.xaxis.set_major_formatter(ScalarFormatter())
    axis.set_ylim(-0.025, 1.0)
    axis.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
a4_output = PLOT_DIR / 'figure_08_spectral_estimators.pdf'
save_pdf(fig_a4, 'figure_08_spectral_estimators.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"Error bars: {A4_ERROR.upper()}. Omitted {a4_omitted} ineligible simulation groups.")
print("Failed/imputed reconstruction theory omitted. Hollow AMP markers contain archived capped runs.")


## Figure 9: Dynamic masking versus full reconstruction

Four activations, d=2000 and 30 ERM seeds. Colored theory curves appear only in the dynamic panel. Green lines are Bayes state evolution; diamonds are the same teacher-initialized AMP reference used in Figure 3.

Output: `plots/figure_09_dynamic_vs_full.pdf`.


In [ ]:
# Dynamic masking versus full reconstruction: four activations, 30 ERM seeds.
# Saved data only. Run the global style cell first.
# Colored lines: dynamic state evolution. Simulation points are unconnected.
# Sample SD error bars match the current notebook convention.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def plot_dynamic_vs_full_autoencoder():
    full_data = DATA_ROOT / 'synthetic/full_reconstruction'
    output_dir = PLOT_DIR
    se_root = DATA_ROOT / 'synthetic/bounded_alpha'
    emp_root = se_root
    se=pd.read_csv(se_root/'state_evolution.csv');se=se.loc[se.masking_protocol.eq('dynamic') & se.k_views.eq(0)]
    dyn=pd.read_csv(emp_root/'erm_summary.csv');dyn=dyn.loc[dyn.masking_protocol.eq('dynamic') & dyn.k_views.eq(0)]
    full=pd.read_csv(full_data/'full_reconstruction_alpha_summary.csv')
    raw=pd.read_csv(full_data/'full_reconstruction_alpha_per_seed.csv')
    bo=pd.read_csv(se_root/'bayes_optimal.csv').sort_values('alpha')
    amp=pd.read_csv(emp_root/'amp_d4000_oracle_plant0p05_summary.csv').sort_values('alpha')
    assert len(raw)==2040 and len(full)==68 and len(dyn)==68
    assert raw.groupby(['activation','alpha']).seed.apply(lambda x: set(x)==set(range(30))).all()
    assert not raw.duplicated(['activation','alpha','seed']).any()
    assert np.isfinite(raw[['cosine_abs','probe_test_accuracy','pretrain_train_loss','pretrain_test_loss']]).all().all()
    assert raw.d.eq(2000).all() and raw.epochs.eq(4000).all()
    assert raw.rho.isna().all() and raw.mask_dtype.eq('none').all()
    for frame in (full,dyn):assert frame.num_seeds.eq(30).all()
    assert amp.num_seeds.eq(10).all()
    check=raw.groupby(['activation','alpha']).cosine_abs.agg(['mean','std']).sort_index()
    reported=full.set_index(['activation','alpha']).sort_index()
    np.testing.assert_allclose(check['mean'],reported.cosine_abs_mean,rtol=1e-12)
    np.testing.assert_allclose(check['std'],reported.cosine_abs_std,rtol=1e-12)
    styles = [(act, label, ACTIVATION_COLORS[act])
              for act, label in (('linear', 'Linear'), ('relu', 'ReLU'),
                                 ('elu', 'ELU'), ('tanh', 'Tanh'))]
    markersize = 3.8 * MARKER_SCALE
    linewidth = 1.35
    fig, axes = plt.subplots(1, 2, figsize=TWO_PANEL_STYLE["figsize"],
                             sharex=True, sharey=True, constrained_layout=True)
    for act, label, color in styles:
        ds = dyn.loc[dyn.activation.eq(act)].sort_values('alpha')
        fs = full.loc[full.activation.eq(act)].sort_values('alpha')
        curve = se.loc[se.activation.eq(act)].sort_values('alpha')
        np.testing.assert_allclose(ds.alpha, fs.alpha, rtol=1e-12)
        assert len(ds) == len(fs) == 17 and len(curve) == 120
        axes[0].plot(curve.alpha, curve.cosine.abs(), color=color,
                     lw=linewidth, zorder=2)
        for ax, points in zip(axes, (ds, fs)):
            ax.errorbar(points.alpha, points.cosine_abs_mean,
                        yerr=points.cosine_abs_std, color=color, marker='o',
                        linestyle='none', markersize=markersize,
                        markerfacecolor=color, markeredgewidth=0,
                        capsize=2.0, elinewidth=0.9, capthick=0.8, zorder=4)
    for ax, title in zip(axes, ('(a) Dynamic masked autoencoder',
                                '(b) Unmasked autoencoder')):
        ax.plot(bo.alpha, bo.bayes_optimal_cosine, color=BAYES_OPTIMAL_COLOR,
                lw=1.45, zorder=1)
        ax.errorbar(amp.alpha, amp.cosine_abs_mean, yerr=amp.cosine_abs_std,
                    color=AMP_COLOR, marker='D', linestyle='none',
                    markersize=markersize, markerfacecolor=AMP_COLOR,
                    markeredgewidth=0, capsize=2.0, elinewidth=0.9,
                    capthick=0.8, zorder=3)
        ax.set_xscale('log')
        ax.set_xlim(.1, 10.2)
        ax.set_ylim(-.04, 1.04)
        ax.set_xticks([.1, .5, 1, 2, 5, 10])
        ax.set_xticklabels(['0.1', '0.5', '1', '2', '5', '10'])
        ax.set_yticks([0, .25, .5, .75, 1])
        ax.tick_params(labelsize=TWO_PANEL_STYLE['tick'], labelleft=True)
        ax.set_xlabel(r'Sample complexity $\alpha=n/d$',
                      fontsize=TWO_PANEL_STYLE['label'])
        ax.set_title(publication_title(title, 'Bounded noise'),
                     fontsize=TWO_PANEL_STYLE['title'])
    axes[0].set_ylabel(r'Cosine similarity, $\theta_{u^\star}$',
                       fontsize=TWO_PANEL_STYLE['label'])
    handles = [Line2D([], [], color=color, marker='o', linestyle='none',
                      markersize=markersize, markeredgewidth=0, label=label)
               for _, label, color in styles]
    handles += [
        Line2D([], [], color=BAYES_OPTIMAL_COLOR, lw=1.45, label='Bayes-optimal'),
    ]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5, .02),
               ncol=5, fontsize=TWO_PANEL_STYLE['legend'], frameon=False,
               handlelength=1.3, handletextpad=.35, columnspacing=.8)
    # ERM: d=2000, 30 seeds; teacher-initialized AMP: d=4000, 10 seeds.
    # Both sets of simulation error bars are sample SD, not SEM.
    name = 'figure_09_dynamic_vs_full'
    save_pdf(fig, 'figure_09_dynamic_vs_full.pdf', bbox_inches='tight', pad_inches=0.12)
    
    return fig, output_dir / (name + '.pdf')


dynamic_full_comparison_fig, dynamic_full_comparison_output = plot_dynamic_vs_full_autoencoder()
plt.show()


## Figure 10: Centered reconstruction errors

Linear activation at alpha=8 under the bounded Beta law. Losses subtract the zero-decoder contribution and exclude the ridge penalty. Dynamic training-loss estimates use evaluation masks on the training samples.

Output: `plots/figure_10_reconstruction_errors.pdf`.


In [ ]:
# Appendix: centered reconstruction errors versus masking ratio.
# Saved data only; run the global style cell first. Lower is better.
# Delta L = reconstruction loss minus the zero-decoder loss on the same samples/masks.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

if 'TWO_PANEL_STYLE' not in globals():
    raise RuntimeError('Run the first cell to initialize the publication style.')
centered_data = DATA_ROOT / 'synthetic/masking_ratio'
centered_se = pd.read_csv(centered_data / 'reconstruction_state_evolution.csv')
centered_erm = pd.read_csv(centered_data / 'reconstruction_erm_summary.csv')
assert len(centered_se) == 1000 and len(centered_erm) == 55
assert centered_se['status'].eq('converged').all()
assert centered_erm['n_seeds'].eq(30).all()
for frame in (centered_se, centered_erm):
    assert frame['activation'].eq('linear').all()
    assert np.allclose(frame['alpha'], 8) and np.allclose(frame['beta'], 1)
    assert np.allclose(frame['lambda_r'], 1e-4)
assert centered_erm['d'].eq(2000).all()
centered_order = ('1', '2', '4', '8', 'dynamic')
centered_colors = publication_k_colors(centered_order)
centered_markers = ('o',) * len(centered_order)
centered_labels = (r'$K=1$ (static)', r'$K=2$', r'$K=4$', r'$K=8$',
                   r'$K\rightarrow\infty$ (dynamic)')
fig_centered_errors, centered_axes = plt.subplots(1, 2, figsize=TWO_PANEL_STYLE['figsize'],
                                                         constrained_layout=True)
centered_handles = []
for key, marker, label in zip(centered_order, centered_markers, centered_labels):
    se_key = 'dynamic' if key == 'dynamic' else 'K' + key
    theory = centered_se.loc[centered_se['curve'].eq(se_key)].sort_values('rho')
    empirical = centered_erm.loc[centered_erm['k_label'].astype(str).eq(key)].sort_values('rho')
    assert len(theory) == 200 and len(empirical) == 11
    color = centered_colors[key]
    for ax, split in zip(centered_axes, ('train', 'test')):
        metric = split + '_centered_loss_per_sample'
        ax.plot(theory['rho'], theory[split + '_centered_loss'], color=color, lw=1.35)
        ax.errorbar(empirical['rho'], empirical[metric + '_mean'],
                    yerr=empirical[metric + '_sd'], fmt=marker, linestyle='none',
                    color=color, mfc=color, ms=3.8*MARKER_SCALE, mew=0,
                    elinewidth=.9, capsize=2.0, capthick=.8, zorder=3)
    centered_handles.append(Line2D([], [], color=color, lw=1.35, label=label))
for ax, split in zip(centered_axes, ('train', 'test')):
    ax.axhline(0, color='0.55', ls='--', lw=.65, zorder=0)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, .2, .4, .6, .8, 1])
    ax.set_xlabel(r'Masking ratio $\rho$', fontsize=TWO_PANEL_STYLE['label'])
    ax.set_ylabel(r'$\Delta\mathcal{L}_{\mathrm{' + split + '}}$', fontsize=TWO_PANEL_STYLE['label'])
    ax.tick_params(labelsize=TWO_PANEL_STYLE['tick'])
fig_centered_errors.legend(handles=centered_handles, loc='upper center',
    bbox_to_anchor=(.5,.02), ncol=5, frameon=False, fontsize=TWO_PANEL_STYLE['legend'],
    handlelength=1.3, handletextpad=.35, columnspacing=.8)
centered_output = PLOT_DIR / 'figure_10_reconstruction_errors.pdf'
save_pdf(fig_centered_errors, 'figure_10_reconstruction_errors.pdf', bbox_inches='tight')
# Lines: replica/state evolution. Markers: ERM means +/- sample SD across 30 seeds.
# alpha=8, beta=1, lambda_R=1e-4, d=2000; gamma=0.25+2.25*Beta(1,2).
plt.show()


## Figure 11: Downstream label-count dependence

Saved state-evolution predictions for linear dynamic masking with n_ds in {1,2,4,8,16,32,64,128}. Reversed magma encodes label count; no new state-evolution computation is performed.

Output: `plots/figure_11_downstream_label_count.pdf`.


In [ ]:
# Linear dynamic masking: downstream label-count sweep, reversed magma colorbar.
# Saved state-evolution predictions only; run the global style cell first.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable


def plot_linear_dynamic_nds():
    source = DATA_ROOT / 'synthetic/downstream_labels/linear_dynamic_nds_state_evolution.csv'
    data = pd.read_csv(source)
    label_counts = np.array([1, 2, 4, 8, 16, 32, 64, 128])
    if set(data['n_ds']) != set(label_counts):
        raise ValueError('Unexpected downstream label counts.')
    # Reversed magma; omit its palest 10% for visibility on a white background.
    cmap = LinearSegmentedColormap.from_list(
        'magma_r_readable', plt.get_cmap('magma_r')(np.linspace(0.10, 1.0, 256)))
    norm = LogNorm(vmin=1, vmax=128)
    fig, ax = plt.subplots(
        figsize=(TWO_PANEL_STYLE["figsize"][0] / 2, TWO_PANEL_STYLE["figsize"][1]),
        constrained_layout=True)
    for n_ds in label_counts:
        curve = data.loc[data['n_ds'].eq(n_ds)].sort_values('alpha')
        if len(curve) != 120 or not np.isfinite(curve['downstream_test_accuracy']).all():
            raise ValueError(f'Incomplete accuracy curve for n_ds={n_ds}.')
        ax.plot(curve['alpha'], curve['downstream_test_accuracy'],
                color=cmap(norm(n_ds)), linewidth=1.35)
    ax.set_xscale('log')
    ax.set_xlim(0.4, 10.2)
    ax.set_ylim(0.495, 0.765)
    ax.set_xticks([0.5, 1, 2, 5, 10])
    ax.set_xticklabels(['0.5', '1', '2', '5', '10'])
    ax.set_yticks([0.5, 0.55, 0.6, 0.65, 0.7, 0.75])
    ax.tick_params(labelsize=TWO_PANEL_STYLE['tick'])
    ax.set_xlabel(r'Sample complexity $\alpha=n/d$', fontsize=TWO_PANEL_STYLE['label'])
    ax.set_ylabel('Classification accuracy', fontsize=TWO_PANEL_STYLE['label'])
    colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax,
                           pad=0.025, fraction=0.04, aspect=25, ticks=label_counts)
    colorbar.ax.set_yticklabels([str(n) for n in label_counts])
    colorbar.minorticks_off()
    colorbar.ax.tick_params(labelsize=TWO_PANEL_STYLE['tick'])
    colorbar.set_label(r'$n_{\rm ds}$', fontsize=TWO_PANEL_STYLE['label'])
    # beta=1, rho=0.75, lambda_r=1e-4; bounded Beta noise.
    output = PLOT_DIR / 'figure_11_downstream_label_count.pdf'
    save_pdf(fig, 'figure_11_downstream_label_count.pdf', dpi=300, bbox_inches='tight', pad_inches=0.12)
    
    return fig, output


linear_dynamic_nds_fig, linear_dynamic_nds_output = plot_linear_dynamic_nds()
plt.show()


## Figure 12: CNN full-reconstruction training and probing

Five encoder/split seeds. Reconstruction MSE is shown for training and held-out validation; downstream accuracy uses the official test set. The marked checkpoint is selected by downstream validation accuracy.

Output: `plots/figure_12_cnn_full_reconstruction.pdf`.


In [ ]:
# Uncertainty: plotted error bars and shaded bands are sample standard deviation (SD).
# Run the shared style cell first. No external plotting scripts are called.
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

a1_dir = DATA_ROOT / 'vision/cnn/full_reconstruction'
a1_probe = pd.read_csv(a1_dir / "linear_probe_only_plot_data.csv").sort_values("pretraining_epoch")
a1_loss = pd.read_csv(a1_dir / "pretraining_loss_aggregate.csv").sort_values("pretraining_epoch")
for frame, cols in (
    (a1_probe, ["test_accuracy_mean", "test_accuracy_std"]),
    (a1_loss, ["train_loss_mean", "train_loss_std", "validation_loss_mean", "validation_loss_std"]),
):
    if frame["pretraining_epoch"].duplicated().any():
        raise ValueError("Expected one aggregate row per pretraining epoch.")
    if not np.isfinite(frame[cols].to_numpy(float)).all():
        raise ValueError("Missing or nonfinite plotting data.")
    if (frame.filter(regex="_std$") < 0).any().any():
        raise ValueError("Negative sample standard deviation.")
    if not frame["n_independent_pretraining_seeds"].eq(5).all():
        raise ValueError("Expected five independent seeds.")
    for std_col in (c for c in cols if c.endswith("_std")):
        sem_col = std_col.replace("_std", "_sem")
        if not np.allclose(frame[sem_col], frame[std_col] / np.sqrt(5)):
            raise ValueError("Inconsistent SD/SEM columns.")
a1_selected = a1_probe.loc[
    a1_probe["selected_by_validation"].astype(str).str.lower().eq("true")
]
if len(a1_selected) != 1:
    raise ValueError("Expected exactly one validation-selected checkpoint.")
a1_selected_epoch = int(a1_selected["pretraining_epoch"].iloc[0])

fig_a1, axes_a1 = plt.subplots(1, 2, figsize=TWO_PANEL_STYLE["figsize"])
# Right: frozen linear-probe test accuracy; SD, no connecting lines.
axes_a1[1].errorbar(
    a1_probe["pretraining_epoch"], a1_probe["test_accuracy_mean"],
    yerr=a1_probe["test_accuracy_std"], fmt="o", linestyle="none",
    color=NO_MASKING_COLOR, markerfacecolor=NO_MASKING_COLOR,
    markeredgewidth=0, markersize=3.8 * MARKER_SCALE,
    capsize=2.0, capthick=0.8, elinewidth=0.9, zorder=3,
)
axes_a1[1].set_title(publication_title("(b) Frozen linear probing", "CIFAR-10"),
                     fontsize=TWO_PANEL_STYLE["title"])
axes_a1[1].set_ylabel("Classification accuracy", fontsize=TWO_PANEL_STYLE["label"])

# Left: reconstruction MSE; same color denotes full reconstruction.
# Solid/dashed distinguish train/held-out validation; bands are SD.
for prefix, label, linestyle in (
    ("train", "Training", "-"), ("validation", "Validation", "--"),
):
    mean = a1_loss[prefix + "_loss_mean"].to_numpy(float)
    sd = a1_loss[prefix + "_loss_std"].to_numpy(float)
    if np.any(mean - sd <= 0):
        raise ValueError("A nonpositive SD band cannot be displayed on a log axis.")
    axes_a1[0].plot(a1_loss["pretraining_epoch"], mean,
                    color=NO_MASKING_COLOR, linestyle=linestyle, linewidth=1.35,
                    label=label)
    axes_a1[0].fill_between(a1_loss["pretraining_epoch"], mean - sd, mean + sd,
                            color=NO_MASKING_COLOR, alpha=0.14, linewidth=0)
axes_a1[0].set_yscale("log")
axes_a1[0].set_title(publication_title("(a) Full reconstruction", "CIFAR-10"),
                     fontsize=TWO_PANEL_STYLE["title"])
axes_a1[0].set_ylabel("Reconstruction MSE", fontsize=TWO_PANEL_STYLE["label"])
axes_a1[0].legend(loc="upper right", frameon=False, fontsize=TWO_PANEL_STYLE["legend"])

for axis in axes_a1:
    axis.set_xscale("log")
    axis.set_xlabel("Pretraining epoch", fontsize=TWO_PANEL_STYLE["label"])
    axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])
    axis.axvline(a1_selected_epoch, color="0.35", linestyle=":", linewidth=1.0, zorder=1)
    axis.set_xlim(0.85, 950)
    axis.set_xticks([1, 4, 16, 64, 256, 800])
    axis.xaxis.set_major_formatter(ScalarFormatter())
fig_a1.legend(
    handles=[Line2D([], [], color="0.35", linestyle=":", linewidth=1.0,
                    label=f"Validation-selected checkpoint: epoch {a1_selected_epoch}")],
    loc="upper center", bbox_to_anchor=(0.5, 0.02), frameon=False,
    fontsize=TWO_PANEL_STYLE["legend"],
)

# EDIT LIMITS AND Y TICKS HERE.
axes_a1[1].set_ylim(0.30, 0.45)
axes_a1[1].set_yticks([0.30, 0.35, 0.40, 0.45])
# axes_a1[0].set_ylim(1e-3, 1.0)
# axes_a1[0].set_yticks([1e-3, 1e-2, 1e-1, 1.0])

fig_a1.tight_layout(pad=0.5, w_pad=1.2)
a1_output = PLOT_DIR / 'figure_12_cnn_full_reconstruction.pdf'
save_pdf(fig_a1, 'figure_12_cnn_full_reconstruction.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 13: ViT full-reconstruction training and probing

Single-seed full-reconstruction ViT run. The right panel shows frozen-probe top-5 validation accuracy. The marked checkpoint maximizes that validation metric among the supplied checkpoints; no seed uncertainty is available.

Output: `plots/figure_13_vit_full_reconstruction.pdf`.


In [ ]:
# Appendix: no-masking ViT, reconstruction and frozen linear probing.
# Shared style only; all plotting is inline and inputs are CSV files.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib.lines import Line2D

av_dir = DATA_ROOT / 'vision/vit/full_reconstruction'
av_data = pd.read_csv(av_dir / "full_reconstruction_train_val_loss_and_linear_probe_top5_vs_pretraining_epochs.csv").sort_values("completed_pretraining_epochs")
av_probe = pd.read_csv(av_dir / "full_reconstruction_linear_probe_accuracy_vs_pretraining_epoch_early_recovery.csv").sort_values("completed_pretraining_epochs")
av_cols = ["completed_pretraining_epochs", "pretraining_train_reconstruction_mse",
           "pretraining_val_reconstruction_mse", "final_acc5"]
if av_data["completed_pretraining_epochs"].duplicated().any() or not np.isfinite(av_data[av_cols]).all().all():
    raise ValueError("Duplicate epochs or missing plotting data.")
if not av_data["number_of_seeds"].eq(1).all():
    raise ValueError("Revisit uncertainty handling: expected the supplied single-seed data.")
if not np.array_equal(av_data["completed_pretraining_epochs"], av_probe["completed_pretraining_epochs"]) or not np.allclose(av_data["final_acc5"], av_probe["final_acc5"]):
    raise ValueError("The two exports disagree on linear-probe accuracy.")
if not av_data["completed_pretraining_epochs"].eq(av_data["pretraining_epoch"] + 1).all():
    raise ValueError("Unexpected epoch indexing.")
av_epochs = av_data["completed_pretraining_epochs"]
av_best = av_data.loc[av_data["final_acc5"].idxmax()]
av_best_epoch = int(av_best["completed_pretraining_epochs"])
av_best_checkpoint = int(av_best["pretraining_epoch"])
fig_av, axes_av = plt.subplots(1, 2, figsize=TWO_PANEL_STYLE["figsize"])
for column, label, linestyle in (
    ("pretraining_train_reconstruction_mse", "Training", "-"),
    ("pretraining_val_reconstruction_mse", "Validation", "--"),
):
    if (av_data[column] <= 0).any():
        raise ValueError("Logarithmic MSE requires positive values.")
    axes_av[0].plot(av_epochs, av_data[column], color=NO_MASKING_COLOR,
                    linestyle=linestyle, linewidth=1.35, label=label)
axes_av[0].set_yscale("log")
axes_av[0].set_title(publication_title("(a) Full reconstruction", "ImageNet-100"),
                     fontsize=TWO_PANEL_STYLE["title"])
axes_av[0].set_ylabel("Reconstruction MSE", fontsize=TWO_PANEL_STYLE["label"])
axes_av[0].legend(loc="upper right", frameon=False, fontsize=TWO_PANEL_STYLE["legend"])
# Final probe epoch (49), not the maximum accuracy across probe epochs.
# One seed: neither SD nor SD is estimable from this export.
axes_av[1].plot(av_epochs, av_data["final_acc5"], "o", linestyle="none",
                color=NO_MASKING_COLOR, markerfacecolor=NO_MASKING_COLOR,
                markeredgewidth=0, markersize=3.8 * MARKER_SCALE, zorder=3)
axes_av[1].set_title(publication_title("(b) Frozen linear probing (top-5)", "ImageNet-100"),
                     fontsize=TWO_PANEL_STYLE["title"])
axes_av[1].set_ylabel("Classification accuracy", fontsize=TWO_PANEL_STYLE["label"])
for axis in axes_av:
    axis.axvline(av_best_epoch, color="0.35", linestyle=":", linewidth=1.0, zorder=1)
    axis.set_xscale("log")
    axis.set_xlabel("Pretraining epoch", fontsize=TWO_PANEL_STYLE["label"])
    axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])
    axis.set_xlim(0.85, 32)
    axis.set_xticks([1, 2, 4, 8, 16, 27])
    axis.xaxis.set_major_formatter(ScalarFormatter())
# EDIT LIMITS AND Y TICKS HERE.
axes_av[0].set_ylim(0.002, 1.0)
axes_av[0].set_yticks([0.01, 0.1, 1.0])
axes_av[1].set_ylim(0.28, 0.41)
axes_av[1].set_yticks([0.28, 0.32, 0.36, 0.40])
fig_av.legend(
    handles=[Line2D([], [], color="0.35", linestyle=":", linewidth=1.0,
                    label=f"Validation-selected checkpoint: epoch {av_best_epoch}")],
    loc="upper center", bbox_to_anchor=(0.5, 0.02), frameon=False,
    fontsize=TWO_PANEL_STYLE["legend"],
)
fig_av.tight_layout(pad=0.5, w_pad=1.2)
av_output = PLOT_DIR / 'figure_13_vit_full_reconstruction.pdf'
save_pdf(fig_av, 'figure_13_vit_full_reconstruction.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("Single seed: no error bars. Epochs count completed pretraining epochs.")


## Figure 14: CNN augmentation comparison

All 11 masking ratios and five seeds per configuration. Test accuracy is converted from percent to a fraction before averaging. Cropping/flipping and masking conditions follow the ViT comparison.

Output: `plots/figure_14_cnn_augmentation.pdf`.


In [ ]:
# CNN augmentation comparison: mean +/- SD across independent seeds.
# CSV input only; shared publication style matches the ViT comparison.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cnn_aug_raw = pd.read_csv(DATA_ROOT / 'vision/cnn/augmentation/per_run_test_accuracy.csv')
cnn_aug_keys = ["masking", "augmentation", "rho"]
if cnn_aug_raw.duplicated(cnn_aug_keys + ["seed"]).any():
    raise ValueError("Duplicate method/augmentation/rho/seed rows.")
if set(cnn_aug_raw["masking"]) != {"static", "dynamic"} or set(cnn_aug_raw["augmentation"]) != {"none", "crop_flip"}:
    raise ValueError("Unexpected masking or augmentation settings.")
if not np.isfinite(cnn_aug_raw[["rho", "test_accuracy_pct", "test_correct", "test_total"]]).all().all():
    raise ValueError("Nonfinite CNN data.")
cnn_aug_raw["accuracy"] = cnn_aug_raw["test_accuracy_pct"] / 100
if not np.allclose(cnn_aug_raw["accuracy"], cnn_aug_raw["test_correct"] / cnn_aug_raw["test_total"]):
    raise ValueError("Accuracy disagrees with test counts.")
cnn_aug_summary = cnn_aug_raw.groupby(cnn_aug_keys, as_index=False).agg(
    mean=("accuracy", "mean"), sd=("accuracy", "std"), n=("seed", "nunique"))
if len(cnn_aug_summary) != 44 or not cnn_aug_summary["n"].eq(5).all():
    raise ValueError("Expected 44 groups with five independent seeds each.")

fig_cnn_aug, ax_cnn_aug = plt.subplots(
    figsize=(TWO_PANEL_STYLE["figsize"][0] / 2, TWO_PANEL_STYLE["figsize"][1]),
    constrained_layout=True,
)
for mode, color in (("dynamic", DYNAMIC_MASKING_COLOR), ("static", STATIC_MASKING_COLOR)):
    for augmentation, marker in (("none", "o"), ("crop_flip", "x")):
        rows = cnn_aug_summary.loc[cnn_aug_summary["masking"].eq(mode)
                                  & cnn_aug_summary["augmentation"].eq(augmentation)].sort_values("rho")
        ax_cnn_aug.errorbar(rows["rho"], rows["mean"], yerr=rows["sd"],
                            fmt="none", ecolor=color, capsize=2.0,
                            elinewidth=0.9, capthick=0.8, zorder=3)
        if marker == "o":
            ax_cnn_aug.scatter(rows["rho"], rows["mean"], facecolors="white",
                               edgecolors=color, marker=marker, s=(3.8 * MARKER_SCALE)**2,
                               linewidths=1.4, zorder=4)
        else:
            ax_cnn_aug.scatter(rows["rho"], rows["mean"], color=color,
                               marker=marker, s=(3.8 * MARKER_SCALE)**2,
                               linewidths=1.4, zorder=4)
ax_cnn_aug.set_title(publication_title("CNN Autoencoder", "CIFAR-10"), fontsize=TWO_PANEL_STYLE["title"])
ax_cnn_aug.set_ylabel("Classification accuracy", fontsize=TWO_PANEL_STYLE["label"])
ax_cnn_aug.set_xlabel(r"Masking ratio $\rho$", fontsize=TWO_PANEL_STYLE["label"])
ax_cnn_aug.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax_cnn_aug.set_xlim(0.0, 1.0)
ax_cnn_aug.tick_params(axis="both", labelsize=TWO_PANEL_STYLE["tick"])
# Independent color and shape keys avoid repeating all four combinations.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
cnn_aug_handles = [
    Patch(facecolor=DYNAMIC_MASKING_COLOR, edgecolor="none", label="Dynamic"),
    Patch(facecolor=STATIC_MASKING_COLOR, edgecolor="none", label="Static"),
    Line2D([], [], color="black", marker="o", linestyle="none",
           markersize=3.8 * MARKER_SCALE, markerfacecolor="white", markeredgewidth=1.4, label="Without cropping/flipping"),
    Line2D([], [], color="black", marker="x", linestyle="none",
           markersize=3.8 * MARKER_SCALE, markeredgewidth=1.4, label="With cropping/flipping"),
]
fig_cnn_aug.legend(
    handles=cnn_aug_handles, loc="upper center",
    bbox_to_anchor=(0.5, 0.03), ncol=2, fontsize=TWO_PANEL_STYLE["legend"],
    frameon=False, handlelength=1.3, handletextpad=0.35, columnspacing=0.8,
)

# EDIT Y AXIS HERE
ax_cnn_aug.set_yticks([0.40, 0.45, 0.50, 0.55, 0.60])
ax_cnn_aug.set_ylim(0.38, 0.61)

cnn_aug_output = PLOT_DIR / 'figure_14_cnn_augmentation.pdf'
save_pdf(fig_cnn_aug, 'figure_14_cnn_augmentation.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 15: CNN training with plateau learning-rate scheduling

Original ReduceLROnPlateau protocol at rho=0.625, five seeds. Each finite-K epoch traverses its fixed image-mask bank; a dynamic epoch groups 32 presentations per image. Axes use optimizer updates. Validation-selected encoders are evaluated with the frozen probe.

Output: `plots/figure_15_cnn_plateau_training.pdf`.


In [ ]:
# CNN original protocol: training, validation and official-test downstream accuracy.
# Saved data only. Run the global style cell first; no training or external scripts.
# Original ReduceLROnPlateau runs, separate from the preceding fixed-LR control.
# Mean +/- SD across five seeds; accuracy is a fraction, matching the previous cell.
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import FormatStrFormatter, LogFormatterMathtext, NullLocator

cnn_plateau_style_names = ("publication_k_colors", "publication_title",
                        "TWO_PANEL_STYLE", "FONT_SCALE", "MARKER_SCALE")
if any(name not in globals() for name in cnn_plateau_style_names):
    raise RuntimeError("Run the first notebook cell to initialize the shared publication style.")

cnn_plateau_dir = DATA_ROOT / 'vision/cnn/plateau_lr'
cnn_plateau_manifest = json.loads((cnn_plateau_dir / "manifest.json").read_text())
for cnn_plateau_name, cnn_plateau_info in cnn_plateau_manifest["files"].items():
    if hashlib.sha256((cnn_plateau_dir / cnn_plateau_name).read_bytes()).hexdigest() != cnn_plateau_info["sha256"]:
        raise ValueError(f"Input changed: {cnn_plateau_name}; update its data manifest before plotting.")

cnn_plateau_history = pd.read_csv(cnn_plateau_dir / "training_history.csv")
cnn_plateau_checkpoints = pd.read_csv(cnn_plateau_dir / "selected_checkpoints.csv")
cnn_plateau_test = pd.read_csv(cnn_plateau_dir / "test_accuracy.csv")
cnn_plateau_order = ("K1", "K2", "K4", "K8", "K16", "K32", "dynamic")
cnn_plateau_colors = publication_k_colors(cnn_plateau_order)
cnn_plateau_labels = {m: ("Dynamic" if m == "dynamic" else rf"$K={m[1:]}$")
                   for m in cnn_plateau_order}
cnn_plateau_expected = {(m, seed) for m in cnn_plateau_order for seed in range(5)}
for cnn_plateau_frame in (cnn_plateau_history, cnn_plateau_checkpoints, cnn_plateau_test):
    if set(zip(cnn_plateau_frame["masking"], cnn_plateau_frame["seed"])) != cnn_plateau_expected:
        raise ValueError("Expected every masking condition with seeds 0--4.")
if cnn_plateau_history.duplicated(["masking", "seed", "optimizer_updates"]).any():
    raise ValueError("Duplicate condition/seed/update measurements.")
for cnn_plateau_frame in (cnn_plateau_checkpoints, cnn_plateau_test):
    if cnn_plateau_frame.duplicated(["masking", "seed"]).any():
        raise ValueError("Duplicate selected checkpoints or test results.")
if not np.isclose(float(cnn_plateau_manifest["rho"]), 0.625):
    raise ValueError("Expected rho=0.625.")
if not cnn_plateau_test["test_total"].eq(10000).all():
    raise ValueError("Expected the official 10000-image CIFAR-10 test set.")
cnn_plateau_test["accuracy"] = cnn_plateau_test["test_accuracy"]
if not np.isfinite(cnn_plateau_test["accuracy"]).all() or not cnn_plateau_test["accuracy"].between(0, 1).all():
    raise ValueError("Expected finite accuracy fractions between zero and one.")
if not cnn_plateau_test["downstream_stage"].eq("linear_probe").all():
    raise ValueError("Expected frozen linear-probe results.")
cnn_plateau_conditions = cnn_plateau_checkpoints.set_index(["masking", "seed"])["condition"]
if not cnn_plateau_test.set_index(["masking", "seed"])["condition"].sort_index().equals(cnn_plateau_conditions.sort_index()):
    raise ValueError("Test results do not match the pretraining runs.")

# Layout and sizes inherit the notebook's shared controls.
cnn_plateau_width, cnn_plateau_height = TWO_PANEL_STYLE["figsize"]
fig_cnn_plateau, axes_cnn_plateau = plt.subplots(
    1, 3, figsize=(cnn_plateau_width, cnn_plateau_height + 0.2),
    gridspec_kw={"width_ratios": [1, 1, 1]},
)
fig_cnn_plateau.subplots_adjust(left=0.07, right=0.99, bottom=0.25, top=0.91, wspace=0.36)
for axis in axes_cnn_plateau:
    axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])
for axis in axes_cnn_plateau[:2]:
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlim(60, 320000)
    axis.set_xticks([100, 1000, 10000, 100000])
    axis.xaxis.set_major_formatter(LogFormatterMathtext(base=10))
    axis.xaxis.set_minor_locator(NullLocator())
    axis.yaxis.set_major_formatter(FormatStrFormatter("%g"))
    axis.yaxis.set_minor_locator(NullLocator())
    axis.set_xlabel("Optimizer updates", fontsize=TWO_PANEL_STYLE["label"])
axes_cnn_plateau[0].set_title(publication_title("(a) Training", "CNN, CIFAR-10"),
                            fontsize=TWO_PANEL_STYLE["title"])
axes_cnn_plateau[1].set_title(publication_title("(b) Validation", "CNN, CIFAR-10"),
                            fontsize=TWO_PANEL_STYLE["title"])
axes_cnn_plateau[2].set_title(publication_title("(c) Downstream", "CIFAR-10"),
                            fontsize=TWO_PANEL_STYLE["title"])
axes_cnn_plateau[0].set_ylabel("Reconstruction MSE", fontsize=TWO_PANEL_STYLE["label"])

# EDIT AXIS LIMITS AND TICKS HERE.
axes_cnn_plateau[0].set_ylim(0.165, 0.81)
axes_cnn_plateau[0].set_yticks([0.2, 0.3, 0.5, 0.8])
axes_cnn_plateau[1].set_ylim(0.185, 0.525)
axes_cnn_plateau[1].set_yticks([0.2, 0.25, 0.3, 0.4, 0.5])
axes_cnn_plateau[2].set_ylim(0.40, 0.61)
axes_cnn_plateau[2].set_yticks([0.40, 0.45, 0.50, 0.55, 0.60])

cnn_plateau_zoom = axes_cnn_plateau[1].inset_axes([0.54, 0.68, 0.4, 0.27])
cnn_plateau_zoom.set_xscale("log")
cnn_plateau_zoom.set_yscale("log")
cnn_plateau_zoom.set_xlim(20000, 300000)
cnn_plateau_zoom.set_ylim(0.193, 0.225)
cnn_plateau_zoom.set_xticks([100000])
cnn_plateau_zoom.xaxis.set_major_formatter(LogFormatterMathtext(base=10))
cnn_plateau_zoom.xaxis.set_minor_locator(NullLocator())
cnn_plateau_zoom.set_yticks([0.20, 0.21, 0.22])
cnn_plateau_zoom.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
cnn_plateau_zoom.yaxis.set_minor_locator(NullLocator())
cnn_plateau_zoom.tick_params(which="both", labelsize=0.68 * TWO_PANEL_STYLE["tick"],
                           length=1.5, pad=1)

def cnn_plateau_plot_loss(axis, frame, value, color, linestyle="-", selected=None, inset=False):
    # Raw seed traces continue to their actual endpoints.
    for _, rows in frame.groupby("seed"):
        rows = rows.sort_values("optimizer_updates")
        axis.plot(rows["optimizer_updates"], rows[value], color=color,
                  linestyle=linestyle, linewidth=0.5, alpha=0.23, zorder=1)
    # Only average updates observed for all five seeds; never pad stopped runs.
    common = frame.pivot(index="optimizer_updates", columns="seed", values=value).dropna()
    if list(common.columns) != list(range(5)) or common.empty:
        raise ValueError("Expected five seeds on a common update grid.")
    values = common.to_numpy(float)
    if not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("Logarithmic loss curves require positive finite measurements.")
    mean = values.mean(axis=1)
    sd = values.std(axis=1, ddof=1)
    axis.fill_between(common.index, mean-sd, mean+sd, color=color,
                      alpha=0.18, linewidth=0, zorder=2)
    axis.plot(common.index, mean, color=color, linestyle=linestyle,
              linewidth=1.0 if inset else 1.25, zorder=3)
    if selected is not None:
        for _, point in selected.iterrows():
            one = frame.loc[frame["seed"].eq(point["seed"])].set_index("optimizer_updates")
            size = (1.6 if inset else 2.3) * MARKER_SCALE
            axis.plot(point["selected_updates"], one.loc[point["selected_updates"], value],
                      marker="o", linestyle="none", color=color, markerfacecolor="white",
                      markeredgewidth=0.6, markersize=size, zorder=5)
            axis.plot(point["stopping_updates"], one.loc[point["stopping_updates"], value],
                      marker="x", linestyle="none", color=color, markeredgewidth=0.6,
                      markersize=size, zorder=5)

for cnn_plateau_method in cnn_plateau_order:
    cnn_plateau_color = cnn_plateau_colors[cnn_plateau_method]
    cnn_plateau_one = cnn_plateau_history.loc[cnn_plateau_history["masking"].eq(cnn_plateau_method)]
    cnn_plateau_selected = cnn_plateau_checkpoints.loc[
        cnn_plateau_checkpoints["masking"].eq(cnn_plateau_method)]
    cnn_plateau_plot_loss(axes_cnn_plateau[0], cnn_plateau_one, "train_loss", cnn_plateau_color)
    cnn_plateau_plot_loss(axes_cnn_plateau[1], cnn_plateau_one, "validation_loss",
                       cnn_plateau_color, selected=cnn_plateau_selected)
    cnn_plateau_plot_loss(cnn_plateau_zoom, cnn_plateau_one, "validation_loss",
                       cnn_plateau_color, selected=cnn_plateau_selected, inset=True)

# Official-test accuracy as a fraction, matching the notebook's accuracy convention.
cnn_plateau_test_summary = cnn_plateau_test.groupby("masking").agg(
    mean=("accuracy", "mean"), sd=("accuracy", "std"), n=("seed", "nunique"))
if not cnn_plateau_test_summary["n"].eq(5).all():
    raise ValueError("Expected five test-evaluated encoder/probe pairs per method.")
cnn_plateau_dynamic = cnn_plateau_test_summary.loc["dynamic"]
axes_cnn_plateau[2].axhspan(
    cnn_plateau_dynamic["mean"]-cnn_plateau_dynamic["sd"],
    cnn_plateau_dynamic["mean"]+cnn_plateau_dynamic["sd"],
    color=cnn_plateau_colors["dynamic"], alpha=0.16, linewidth=0)
axes_cnn_plateau[2].axhline(cnn_plateau_dynamic["mean"], color=cnn_plateau_colors["dynamic"],
                         linestyle="--", linewidth=1.25)
cnn_plateau_k = np.array([1, 2, 4, 8, 16, 32])
axes_cnn_plateau[2].plot(cnn_plateau_k, cnn_plateau_test_summary.loc[list(cnn_plateau_order[:-1]), "mean"],
                     color="0.55", linewidth=0.8, zorder=1)
for cnn_plateau_method, cnn_plateau_count in zip(cnn_plateau_order[:-1], cnn_plateau_k):
    cnn_plateau_point = cnn_plateau_test_summary.loc[cnn_plateau_method]
    cnn_plateau_color = cnn_plateau_colors[cnn_plateau_method]
    cnn_plateau_seeds = cnn_plateau_test.loc[cnn_plateau_test["masking"].eq(cnn_plateau_method)].sort_values("seed")
    axes_cnn_plateau[2].scatter(
        cnn_plateau_count * 2.0 ** np.linspace(-0.06, 0.06, 5),
        cnn_plateau_seeds["accuracy"], color=cnn_plateau_color,
        s=(2.1 * MARKER_SCALE)**2, alpha=0.25, linewidths=0, zorder=2)
    axes_cnn_plateau[2].errorbar(
        cnn_plateau_count, cnn_plateau_point["mean"], yerr=cnn_plateau_point["sd"],
        fmt="o", color=cnn_plateau_color, markersize=3.8 * MARKER_SCALE,
        markeredgewidth=0, capsize=2.0, elinewidth=0.9, zorder=3)
axes_cnn_plateau[2].set_xscale("log", base=2)
axes_cnn_plateau[2].set_xlim(0.9, 35)
axes_cnn_plateau[2].set_xticks(cnn_plateau_k)
axes_cnn_plateau[2].xaxis.set_major_formatter(FormatStrFormatter("%g"))
axes_cnn_plateau[2].xaxis.set_minor_locator(NullLocator())
axes_cnn_plateau[2].set_xlabel(r"Number of masks $K$", fontsize=TWO_PANEL_STYLE["label"])
axes_cnn_plateau[2].set_ylabel("Test accuracy", fontsize=TWO_PANEL_STYLE["label"], labelpad=2)

cnn_plateau_method_handles = [
    Line2D([], [], color=cnn_plateau_colors[m], linewidth=1.4, label=cnn_plateau_labels[m])
    for m in cnn_plateau_order]
fig_cnn_plateau.legend(
    handles=cnn_plateau_method_handles, loc="upper center", bbox_to_anchor=(0.5, 0.015),
    ncol=7, frameon=False, fontsize=TWO_PANEL_STYLE["legend"],
    handlelength=1.5, columnspacing=1.0, handletextpad=0.4)
cnn_plateau_detail_handles = [
    Line2D([], [], color="0.4", linewidth=0.5, alpha=0.5),
    (Patch(facecolor="0.6", alpha=0.25, edgecolor="none"),
     Line2D([], [], color="0.3", linewidth=1.25)),
    Line2D([], [], color="0.4", marker="o", markerfacecolor="white", linestyle="none",
           markersize=2.3 * MARKER_SCALE, markeredgewidth=0.6),
    Line2D([], [], color="0.4", marker="x", linestyle="none",
           markersize=2.3 * MARKER_SCALE, markeredgewidth=0.6)]
fig_cnn_plateau.legend(
    handles=cnn_plateau_detail_handles,
    labels=["Individual seeds", r"Mean $\pm$ SD", "Early stopped checkpoint", "Stopping point"],
    handler_map={tuple: HandlerTuple(ndivide=1)}, loc="upper center",
    bbox_to_anchor=(0.5, -0.105), ncol=4, frameon=False,
    fontsize=TWO_PANEL_STYLE["legend"], handlelength=1.5,
    columnspacing=1.0, handletextpad=0.4)

cnn_plateau_output = PLOT_DIR / 'figure_15_cnn_plateau_training.pdf'
save_pdf(fig_cnn_plateau, 'figure_15_cnn_plateau_training.pdf', dpi=300, bbox_inches='tight')

plt.show()

print("rho=0.625; original ReduceLROnPlateau runs; five seeds; mean +/- SD.")
print("Training: preceding-epoch mean MSE; validation: epoch-end MSE.")
print("Dynamic downstream accuracy: dashed horizontal mean +/- SD reference.")
print("Downstream: official-test accuracy of the original validation-selected encoder/probe pairs.")


## Figure 16: CNN training with a fixed learning rate

Constant learning rate 1e-3; rho=0.625 and five seeds. Validation occurs every 2,500 updates; stopping patience is 125,000 updates, with a 1,150,000-update ceiling. Early diagnostic reruns and original histories are separate datasets, kept separate in the plotting code.

Output: `plots/figure_16_cnn_fixed_lr_training.pdf`.


In [ ]:
# CNN fixed-LR control: training, validation and official-test downstream accuracy.
# Saved data only. Run the global style cell first; no training or external scripts.
# Early reruns and original histories use the same solid style, but remain separate datasets.
# Uncertainty: mean +/- SD across five seeds. Test accuracy uses the official test set.
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import FormatStrFormatter, LogFormatterMathtext, NullLocator

cnn_diag_style_names = ("publication_k_colors", "publication_title",
                        "TWO_PANEL_STYLE", "FONT_SCALE", "MARKER_SCALE")
if any(name not in globals() for name in cnn_diag_style_names):
    raise RuntimeError("Run the first notebook cell to initialize the shared publication style.")

cnn_diag_dir = DATA_ROOT / 'vision/cnn/fixed_lr'
cnn_diag_manifest = json.loads((cnn_diag_dir / "manifest.json").read_text())
for cnn_diag_name, cnn_diag_info in cnn_diag_manifest["files"].items():
    if hashlib.sha256((cnn_diag_dir / cnn_diag_name).read_bytes()).hexdigest() != cnn_diag_info["sha256"]:
        raise ValueError(f"Input changed: {cnn_diag_name}; update its data manifest before plotting.")

cnn_diag_history = pd.read_csv(cnn_diag_dir / "training_history.csv")
cnn_diag_early_train = pd.read_csv(cnn_diag_dir / "early_training.csv")
cnn_diag_early_val = pd.read_csv(cnn_diag_dir / "early_validation.csv")
cnn_diag_checkpoints = pd.read_csv(cnn_diag_dir / "selected_checkpoints.csv")
cnn_diag_test = pd.read_csv(cnn_diag_dir / "test_accuracy.csv")
cnn_diag_order = ("K1", "K2", "K4", "K8", "K16", "K32", "dynamic")
cnn_diag_colors = publication_k_colors(cnn_diag_order)
cnn_diag_labels = {m: ("Dynamic" if m == "dynamic" else rf"$K={m[1:]}$")
                   for m in cnn_diag_order}
cnn_diag_expected = {(m, seed) for m in cnn_diag_order for seed in range(5)}
for cnn_diag_frame in (cnn_diag_history, cnn_diag_early_train, cnn_diag_early_val,
                       cnn_diag_checkpoints, cnn_diag_test):
    if set(zip(cnn_diag_frame["masking"], cnn_diag_frame["seed"])) != cnn_diag_expected:
        raise ValueError("Expected every masking condition with seeds 0--4.")
for cnn_diag_frame in (cnn_diag_history, cnn_diag_early_train, cnn_diag_early_val):
    if cnn_diag_frame.duplicated(["masking", "seed", "optimizer_updates"]).any():
        raise ValueError("Duplicate condition/seed/update measurements.")
if not cnn_diag_history["lr"].eq(0.001).all():
    raise ValueError("Expected fixed pretraining learning rate 0.001.")
if not cnn_diag_test["test_total"].eq(10000).all():
    raise ValueError("Expected the official 10000-image CIFAR-10 test set.")
cnn_diag_test["accuracy"] = cnn_diag_test["test_accuracy_pct"] / 100.0
if not np.allclose(cnn_diag_test["accuracy"],
                   cnn_diag_test["test_correct"] / cnn_diag_test["test_total"]):
    raise ValueError("Test accuracy disagrees with saved correct/total counts.")

# Layout and sizes inherit the notebook's shared controls.
cnn_diag_width, cnn_diag_height = TWO_PANEL_STYLE["figsize"]
fig_cnn_diag, axes_cnn_diag = plt.subplots(
    1, 3, figsize=(cnn_diag_width, cnn_diag_height + 0.2),
    gridspec_kw={"width_ratios": [1, 1, 1]},
)
fig_cnn_diag.subplots_adjust(left=0.07, right=0.99, bottom=0.25, top=0.91, wspace=0.36)
for axis in axes_cnn_diag:
    axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])
for axis in axes_cnn_diag[:2]:
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlim(100, 270000)
    axis.set_xticks([100, 1000, 10000, 100000])
    axis.xaxis.set_major_formatter(LogFormatterMathtext(base=10))
    axis.xaxis.set_minor_locator(NullLocator())
    axis.yaxis.set_major_formatter(FormatStrFormatter("%g"))
    axis.yaxis.set_minor_locator(NullLocator())
    axis.set_xlabel("Optimizer updates", fontsize=TWO_PANEL_STYLE["label"])
axes_cnn_diag[0].set_title(publication_title("(a) Training", "CNN, CIFAR-10"),
                            fontsize=TWO_PANEL_STYLE["title"])
axes_cnn_diag[1].set_title(publication_title("(b) Validation", "CNN, CIFAR-10"),
                            fontsize=TWO_PANEL_STYLE["title"])
axes_cnn_diag[2].set_title(publication_title("(c) Downstream Accuracy", "CIFAR-10"),
                            fontsize=TWO_PANEL_STYLE["title"])
axes_cnn_diag[0].set_ylabel("Reconstruction MSE", fontsize=TWO_PANEL_STYLE["label"])

# EDIT AXIS LIMITS AND TICKS HERE.
axes_cnn_diag[0].set_ylim(0.093, 0.81)
axes_cnn_diag[0].set_yticks([0.1, 0.2, 0.5, 0.8])
axes_cnn_diag[1].set_ylim(0.193, 0.525)
axes_cnn_diag[1].set_yticks([0.2, 0.25, 0.3, 0.4, 0.5])
axes_cnn_diag[2].set_ylim(0.40, 0.61)
axes_cnn_diag[2].set_yticks([0.40, 0.45, 0.50, 0.55, 0.60])

cnn_diag_zoom = axes_cnn_diag[1].inset_axes([0.54, 0.68, 0.4, 0.27])
cnn_diag_zoom.set_xscale("log")
cnn_diag_zoom.set_yscale("log")
cnn_diag_zoom.set_xlim(20000, 270000)
cnn_diag_zoom.set_ylim(0.198, 0.225)
cnn_diag_zoom.set_xticks([100000])
cnn_diag_zoom.xaxis.set_major_formatter(LogFormatterMathtext(base=10))
cnn_diag_zoom.xaxis.set_minor_locator(NullLocator())
cnn_diag_zoom.set_yticks([0.20, 0.21, 0.22])
cnn_diag_zoom.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
cnn_diag_zoom.yaxis.set_minor_locator(NullLocator())
cnn_diag_zoom.tick_params(which="both", labelsize=0.68 * TWO_PANEL_STYLE["tick"],
                           length=1.5, pad=1)

def cnn_diag_plot_loss(axis, frame, value, color, linestyle="-", selected=None, inset=False):
    # Raw seed traces continue to their actual endpoints.
    for _, rows in frame.groupby("seed"):
        rows = rows.sort_values("optimizer_updates")
        axis.plot(rows["optimizer_updates"], rows[value], color=color,
                  linestyle=linestyle, linewidth=0.5, alpha=0.23, zorder=1)
    # Only average updates observed for all five seeds; never pad stopped runs.
    common = frame.pivot(index="optimizer_updates", columns="seed", values=value).dropna()
    if list(common.columns) != list(range(5)) or common.empty:
        raise ValueError("Expected five seeds on a common update grid.")
    values = common.to_numpy(float)
    if not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("Logarithmic loss curves require positive finite measurements.")
    mean = values.mean(axis=1)
    sd = values.std(axis=1, ddof=1)
    axis.fill_between(common.index, mean-sd, mean+sd, color=color,
                      alpha=0.18, linewidth=0, zorder=2)
    axis.plot(common.index, mean, color=color, linestyle=linestyle,
              linewidth=1.0 if inset else 1.25, zorder=3)
    if selected is not None:
        for _, point in selected.iterrows():
            one = frame.loc[frame["seed"].eq(point["seed"])].set_index("optimizer_updates")
            size = (1.6 if inset else 2.3) * MARKER_SCALE
            axis.plot(point["selected_updates"], one.loc[point["selected_updates"], value],
                      marker="o", linestyle="none", color=color, markerfacecolor="white",
                      markeredgewidth=0.6, markersize=size, zorder=5)
            axis.plot(point["stopping_updates"], one.loc[point["stopping_updates"], value],
                      marker="x", linestyle="none", color=color, markeredgewidth=0.6,
                      markersize=size, zorder=5)

for cnn_diag_method in cnn_diag_order:
    cnn_diag_color = cnn_diag_colors[cnn_diag_method]
    cnn_diag_one = cnn_diag_history.loc[cnn_diag_history["masking"].eq(cnn_diag_method)]
    cnn_diag_selected = cnn_diag_checkpoints.loc[
        cnn_diag_checkpoints["masking"].eq(cnn_diag_method)]
    cnn_diag_plot_loss(axes_cnn_diag[0], cnn_diag_one, "train_loss", cnn_diag_color)
    cnn_diag_plot_loss(axes_cnn_diag[1], cnn_diag_one, "validation_loss",
                       cnn_diag_color, selected=cnn_diag_selected)
    cnn_diag_plot_loss(cnn_diag_zoom, cnn_diag_one, "validation_loss",
                       cnn_diag_color, selected=cnn_diag_selected, inset=True)
    # Separate reruns: cumulative within-epoch training mean, pointwise validation.
    cnn_diag_plot_loss(
        axes_cnn_diag[0],
        cnn_diag_early_train.loc[cnn_diag_early_train["masking"].eq(cnn_diag_method)],
        "cumulative_mean_train_loss", cnn_diag_color, linestyle="-")
    cnn_diag_plot_loss(
        axes_cnn_diag[1],
        cnn_diag_early_val.loc[cnn_diag_early_val["masking"].eq(cnn_diag_method)
                               & cnn_diag_early_val["optimizer_updates"].gt(0)],
        "validation_mse", cnn_diag_color, linestyle="-")

# Official-test accuracy as a fraction, matching the notebook's accuracy convention.
cnn_diag_test_summary = cnn_diag_test.groupby("masking").agg(
    mean=("accuracy", "mean"), sd=("accuracy", "std"), n=("seed", "nunique"))
if not cnn_diag_test_summary["n"].eq(5).all():
    raise ValueError("Expected five test-evaluated encoder/probe pairs per method.")
cnn_diag_dynamic = cnn_diag_test_summary.loc["dynamic"]
axes_cnn_diag[2].axhspan(
    cnn_diag_dynamic["mean"]-cnn_diag_dynamic["sd"],
    cnn_diag_dynamic["mean"]+cnn_diag_dynamic["sd"],
    color=cnn_diag_colors["dynamic"], alpha=0.16, linewidth=0)
axes_cnn_diag[2].axhline(cnn_diag_dynamic["mean"], color=cnn_diag_colors["dynamic"],
                         linestyle="--", linewidth=1.25)
cnn_diag_k = np.array([1, 2, 4, 8, 16, 32])
axes_cnn_diag[2].plot(cnn_diag_k, cnn_diag_test_summary.loc[list(cnn_diag_order[:-1]), "mean"],
                     color="0.55", linewidth=0.8, zorder=1)
for cnn_diag_method, cnn_diag_count in zip(cnn_diag_order[:-1], cnn_diag_k):
    cnn_diag_point = cnn_diag_test_summary.loc[cnn_diag_method]
    cnn_diag_color = cnn_diag_colors[cnn_diag_method]
    cnn_diag_seeds = cnn_diag_test.loc[cnn_diag_test["masking"].eq(cnn_diag_method)].sort_values("seed")
    axes_cnn_diag[2].scatter(
        cnn_diag_count * 2.0 ** np.linspace(-0.06, 0.06, 5),
        cnn_diag_seeds["accuracy"], color=cnn_diag_color,
        s=(2.1 * MARKER_SCALE)**2, alpha=0.25, linewidths=0, zorder=2)
    axes_cnn_diag[2].errorbar(
        cnn_diag_count, cnn_diag_point["mean"], yerr=cnn_diag_point["sd"],
        fmt="o", color=cnn_diag_color, markersize=3.8 * MARKER_SCALE,
        markeredgewidth=0, capsize=2.0, elinewidth=0.9, zorder=3)
axes_cnn_diag[2].set_xscale("log", base=2)
axes_cnn_diag[2].set_xlim(0.9, 35)
axes_cnn_diag[2].set_xticks(cnn_diag_k)
axes_cnn_diag[2].xaxis.set_major_formatter(FormatStrFormatter("%g"))
axes_cnn_diag[2].xaxis.set_minor_locator(NullLocator())
axes_cnn_diag[2].set_xlabel(r"Number of masks $K$", fontsize=TWO_PANEL_STYLE["label"])
axes_cnn_diag[2].set_ylabel("Test accuracy", fontsize=TWO_PANEL_STYLE["label"], labelpad=2)

cnn_diag_method_handles = [
    Line2D([], [], color=cnn_diag_colors[m], linewidth=1.4, label=cnn_diag_labels[m])
    for m in cnn_diag_order]
fig_cnn_diag.legend(
    handles=cnn_diag_method_handles, loc="upper center", bbox_to_anchor=(0.5, 0.015),
    ncol=7, frameon=False, fontsize=TWO_PANEL_STYLE["legend"],
    handlelength=1.5, columnspacing=1.0, handletextpad=0.4)
cnn_diag_detail_handles = [
    Line2D([], [], color="0.4", linewidth=0.5, alpha=0.5),
    (Patch(facecolor="0.6", alpha=0.25, edgecolor="none"),
     Line2D([], [], color="0.3", linewidth=1.25)),
    Line2D([], [], color="0.4", marker="o", markerfacecolor="white", linestyle="none",
           markersize=2.3 * MARKER_SCALE, markeredgewidth=0.6),
    Line2D([], [], color="0.4", marker="x", linestyle="none",
           markersize=2.3 * MARKER_SCALE, markeredgewidth=0.6)]
fig_cnn_diag.legend(
    handles=cnn_diag_detail_handles,
    labels=["Individual seeds", r"Mean $\pm$ SD", "Early stopped checkpoint", "Stopping point"],
    handler_map={tuple: HandlerTuple(ndivide=1)}, loc="upper center",
    bbox_to_anchor=(0.5, -0.105), ncol=4, frameon=False,
    fontsize=TWO_PANEL_STYLE["legend"], handlelength=1.5,
    columnspacing=1.0, handletextpad=0.4)

cnn_diag_output = PLOT_DIR / 'figure_16_cnn_fixed_lr_training.pdf'
save_pdf(fig_cnn_diag, 'figure_16_cnn_fixed_lr_training.pdf', dpi=300, bbox_inches='tight')

plt.show()

print("rho=0.625; fixed LR=0.001; patience=50; five seeds; mean +/- SD.")
print("Early reruns and original histories share solid styling; data remain separate.")
print("Training: cumulative prefix means before 2500, full-epoch means thereafter.")
print("Downstream: official-test accuracy of the original validation-selected encoder/probe pairs.")


## Figure 17: CNN downstream comparison and fixed-budget control

Panel (a) compares validation-selected encoders. Panel (b) uses separate fixed-budget runs with pretraining early stopping disabled and exactly 125,000 updates for every condition. Five seeds and the same frozen-probe protocol are used.

Output: `plots/figure_17_cnn_budget_control.pdf`.


In [ ]:
# CNN downstream test accuracy: early-stopped versus fixed-update encoders.
# (a) Original plateau-LR and fixed-LR/patience-50 validation-selected checkpoints.
# (b) New fixed-LR encoders trained for exactly 125,000 optimizer updates.
# Saved official-test evaluations only; mean +/- SD across five seeds.
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cnn_compare_required = ("TWO_PANEL_STYLE", "MARKER_SCALE", "publication_title",
                        "DYNAMIC_MASKING_COLOR", "NO_MASKING_COLOR")
if any(name not in globals() for name in cnn_compare_required):
    raise RuntimeError("Run the first cell to initialize the shared publication style.")
cnn_compare_inputs = (
    ("original", "plateau_lr", "test_accuracy.csv", "test_accuracy", 1.0),
    ("fixed_lr_early", "fixed_lr", "test_accuracy.csv", "test_accuracy_pct", 0.01),
    ("fixed_lr_125k", "fixed_budget", "per_run_test_metrics.csv", "test_accuracy_pct", 0.01),
)
cnn_compare_root = DATA_ROOT / "vision/cnn"

cnn_compare_order = ("K1", "K2", "K4", "K8", "K16", "K32", "dynamic")
cnn_compare_expected = {(method, seed) for method in cnn_compare_order for seed in range(5)}
cnn_compare_summaries = {}
for key, directory, filename, column, scale in cnn_compare_inputs:
    folder = cnn_compare_root / directory
    path = folder / filename
    manifest = json.loads((folder / "manifest.json").read_text())
    if hashlib.sha256(path.read_bytes()).hexdigest() != manifest["files"][filename]["sha256"]:
        raise ValueError(f"Input changed: {path}; verify its provenance.")
    frame = pd.read_csv(path)
    if frame.duplicated(["masking", "seed"]).any() or set(zip(frame["masking"], frame["seed"])) != cnn_compare_expected:
        raise ValueError("Expected exactly seven masking settings and seeds 0--4.")
    if not np.allclose(frame["rho"], 0.625) or not frame["test_total"].eq(10000).all():
        raise ValueError("Expected rho=0.625 and the official 10000-image CIFAR-10 test set.")
    if key == "fixed_lr_125k":
        if not frame["optimizer_updates"].eq(125000).all():
            raise ValueError("Panel (b) requires exactly 125,000 pretraining updates.")
        if not frame["checkpoint"].str.endswith("/update_125000.pt").all():
            raise ValueError("Panel (b) must use the fixed-update encoder checkpoints.")
    frame["accuracy"] = frame[column] * scale
    if not np.isfinite(frame["accuracy"]).all() or not frame["accuracy"].between(0, 1).all():
        raise ValueError("Invalid accuracy values.")
    if "test_correct" in frame and not np.allclose(frame["accuracy"], frame["test_correct"] / frame["test_total"]):
        raise ValueError("Accuracy disagrees with saved test counts.")
    summary = frame.groupby("masking").agg(mean=("accuracy", "mean"),
                                           sd=("accuracy", "std"), n=("seed", "nunique"))
    cnn_compare_summaries[key] = summary.loc[list(cnn_compare_order)]

# Preserve the original colors/markers for the two learning-rate protocols.
# The fixed-LR series has the same appearance on both sides; only encoder
# selection and stopping differ: panel (b) uses separate fixed-budget runs.
cnn_compare_styles = {
    "original": (DYNAMIC_MASKING_COLOR, "o", "Original (plateau LR scheduler)"),
    "fixed_lr_early": (NO_MASKING_COLOR, "x", "Fixed LR, 125k-update patience"),
    "fixed_lr_125k": (NO_MASKING_COLOR, "x", "Fixed LR"),
}
cnn_compare_width, cnn_compare_height = TWO_PANEL_STYLE["figsize"]
fig_cnn_compare = plt.figure(figsize=(cnn_compare_width, cnn_compare_height + 0.20))
cnn_compare_grid = fig_cnn_compare.add_gridspec(
    1, 2, left=0.075, right=0.99, bottom=0.28, top=0.84, wspace=0.19)
cnn_compare_x = np.array([1., 2., 4., 8., 16., 32.])
cnn_compare_panels = (
    ("(a) Early-stopped checkpoint", ("original", "fixed_lr_early")),
    ("(b) 125k optimizer updates", ("fixed_lr_125k",)),
)
cnn_compare_axes = []
for panel_index, (title, keys) in enumerate(cnn_compare_panels):
    inner_grid = cnn_compare_grid[panel_index].subgridspec(
        1, 2, width_ratios=[6, 1], wspace=0.08)
    ax = fig_cnn_compare.add_subplot(inner_grid[0],
                                    sharey=cnn_compare_axes[0][0] if cnn_compare_axes else None)
    ax_dynamic = fig_cnn_compare.add_subplot(inner_grid[1], sharey=ax)
    cnn_compare_axes.append((ax, ax_dynamic))
    for key in keys:
        summary = cnn_compare_summaries[key]
        color, marker, label = cnn_compare_styles[key]
        for axis, xs, values in ((ax, cnn_compare_x, summary.iloc[:-1]),
                                 (ax_dynamic, [0.], summary.iloc[-1:])):
            axis.errorbar(xs, values["mean"], yerr=values["sd"],
                          fmt=marker, linestyle="none", color=color,
                          markerfacecolor="white" if marker == "o" else color,
                          markeredgewidth=1.0,
                          markersize=(4.1 if marker == "o" else 3.4) * MARKER_SCALE,
                          capsize=2.5, elinewidth=0.9, capthick=0.8,
                          label=label, zorder=3)
    ax.set_xscale("log", base=2)
    ax.set_xticks(cnn_compare_x, ["1", "2", "4", "8", "16", "32"])
    ax.set_xlim(0.8, 40.)
    ax_dynamic.set_xlim(-0.7, 0.7)
    ax_dynamic.set_xticks([0.], [r"$K\to\infty$" + "\nDynamic"])
    ax.set_ylim(0.40, 0.605)
    ax.set_yticks([0.42, 0.46, 0.50, 0.54, 0.58])
    for axis in (ax, ax_dynamic):
        axis.tick_params(axis="both", labelsize=TWO_PANEL_STYLE["tick"])
        axis.spines["top"].set_visible(True)
        axis.spines["right"].set_visible(False)
    ax_dynamic.spines["right"].set_visible(True)
    ax_dynamic.spines["left"].set_visible(False)
    ax_dynamic.tick_params(axis="y", left=False, right=False, labelleft=False)
    # A gap separates the categorical dynamic limit from the finite-K axis.
    for axis, xedge in ((ax, 1), (ax_dynamic, 0)):
        axis.plot([xedge, xedge], [0, 1], transform=axis.transAxes,
                  marker="|", markersize=8, linestyle="none", color="black",
                  markeredgewidth=0.9, clip_on=False)
    ax.legend(loc="upper left", frameon=False, fontsize=TWO_PANEL_STYLE["legend"],
              handletextpad=0.65, borderaxespad=0.4)
    if panel_index == 0:
        ax.set_ylabel("Test accuracy", fontsize=TWO_PANEL_STYLE["label"])
    else:
        ax.tick_params(axis="y", labelleft=False)
    center = (ax.get_position().x0 + ax_dynamic.get_position().x1) / 2
    fig_cnn_compare.text(center, 0.91, publication_title(title, r"CNN, $\rho=0.625$"),
                         ha="center", va="center", fontsize=TWO_PANEL_STYLE["title"])
    fig_cnn_compare.text(center, 0.035, r"Number of masks $K$",
                         ha="center", va="bottom", fontsize=TWO_PANEL_STYLE["label"])

# Export mean and SD in percent / percentage points alongside the figure.
cnn_compare_table = pd.DataFrame(index=pd.Index(cnn_compare_order, name="masking"))
for key, summary in cnn_compare_summaries.items():
    cnn_compare_table[f"{key}_accuracy_pct"] = 100 * summary["mean"]
    cnn_compare_table[f"{key}_sd_pp"] = 100 * summary["sd"]
cnn_compare_output = PLOT_DIR / 'figure_17_cnn_budget_control.pdf'
save_pdf(fig_cnn_compare, 'figure_17_cnn_budget_control.pdf', dpi=300, bbox_inches='tight')

# The comparison table is displayed below; input data remain unchanged.
plt.show()

print(cnn_compare_table.round(3).to_string())
print("Panels show official-test accuracy: fraction in the plot, percent in the table; errors are SD over five seeds.")
print("Panel (a): validation-selected encoders. Panel (b): exactly 125,000 pretraining updates; fixed LR=0.001.")


## Figure 18: Finite-K threshold ratios

Only homogeneous and bounded matched-moment rows are plotted. Linear activation, zero ridge, beta from 0.1 to 1 and rho in {0.05,0.25,0.5,0.75,0.95}. The normalized thresholds use the finite-beta dynamic denominator. This bounded law differs from the bounded Beta law in the main recovery plots. Additional archived rows remain in the CSV.

Output: `plots/figure_18_finite_k_threshold_ratios.pdf`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, NullFormatter

a5_data = pd.read_csv(DATA_ROOT / 'thresholds/finite_k_ratios' /
                      "validated_ratio_data_matched_three_laws_grid.csv")
A5_BETA_COLORMAP = "magma_r"
A5_RHOS = [0.05, 0.25, 0.5, 0.75, 0.95]
a5_laws = [
    ("homogeneous", "(a) Homogeneous"),
    ("bounded_matched_lognormal_moments", "(b) Bounded"),
]
a5_data = a5_data.loc[a5_data["beta"].between(0.1, 1.0)
                      & a5_data["gamma_dist"].isin([law for law, _ in a5_laws])
                      & np.isfinite(a5_data["K_numeric"])].copy()
if a5_data.duplicated(["gamma_dist", "beta", "rho", "K_numeric"]).any():
    raise ValueError("Duplicate threshold ratios.")
if not a5_data["status"].eq("converged").all():
    raise ValueError("Expected converged saved numerator thresholds.")
if not np.isfinite(a5_data[["threshold_ratio", "alpha_c", "alpha_dynamic"]]).all().all():
    raise ValueError("Missing ratio or threshold.")
if (a5_data["alpha_dynamic"] <= 0).any() or not np.allclose(
    a5_data["threshold_ratio"], a5_data["alpha_c"] / a5_data["alpha_dynamic"]
):
    raise ValueError("Invalid saved normalization.")
a5_betas = sorted(a5_data["beta"].unique(), reverse=True)
a5_norm = Normalize(vmin=0.1, vmax=1.0)
a5_cmap = LinearSegmentedColormap.from_list(
    "a5_beta", plt.get_cmap(A5_BETA_COLORMAP)(np.linspace(0.10, 0.85, 256)))
# Preserve the publication width, fonts and individual panel heights.
fig_a5, axes_a5 = plt.subplots(len(a5_laws), 5, figsize=(TWO_PANEL_STYLE["figsize"][0], 3.3),
                              sharex=True, sharey="col")
a5_k_line = np.geomspace(1, 128, 300)
for i, (law, law_title) in enumerate(a5_laws):
    for j, rho in enumerate(A5_RHOS):
        axis = axes_a5[i, j]
        panel = a5_data.loc[a5_data["gamma_dist"].eq(law) & np.isclose(a5_data["rho"], rho)]
        if panel.empty:
            raise ValueError(f"Missing panel: {law}, rho={rho}")
        axis.plot(a5_k_line, 1 + (1 / (2 * rho * (1-rho)) - 1) / a5_k_line,
                  color="black", linewidth=1.2, zorder=2, alpha=0.8)
        axis.axhline(1.0, color="0.4", linestyle="--", linewidth=0.9, zorder=1)
        axis.grid(False, which="both", axis="both")
        for beta in a5_betas:
            rows = panel.loc[np.isclose(panel["beta"], beta)].sort_values("K_numeric")
            if len(rows) != 8:
                raise ValueError(f"Incomplete K grid: {law}, rho={rho}, beta={beta}")
            axis.scatter(rows["K_numeric"], rows["threshold_ratio"],
                         color=a5_cmap(a5_norm(beta)), marker="o",
                         s=(3.8 * MARKER_SCALE)**2, linewidths=0, zorder=3, alpha=0.8)
        axis.set_xscale("log", base=2)
        axis.set_xlim(0.8, 160)
        axis.set_xticks([1, 8, 128], labels=["1", "8", "128"])
        axis.minorticks_off()
        axis.tick_params(axis="both", which="both", labelsize=TWO_PANEL_STYLE["tick"])
        if i == 0:
            axis.set_title(rf"$\rho={rho:g}$", fontsize=TWO_PANEL_STYLE["title"])
        if i == len(a5_laws) - 1:
            axis.set_xlabel(r"$K$", fontsize=TWO_PANEL_STYLE["label"])
        if j == 4:
            axis.text(1.10, 0.5, law_title, transform=axis.transAxes,
                      rotation=270, ha="left", va="center",
                      fontsize=TWO_PANEL_STYLE["label"])
fig_a5.supylabel(r"$\alpha_c^{(K)}/\alpha_c^{(\infty)}$",
                 x=0.025, y=0.602, fontsize=TWO_PANEL_STYLE["label"])

# EDIT Y LIMITS HERE: one common scale per masking-ratio column.
for j, rho in enumerate(A5_RHOS):
    max_ratio = a5_data.loc[np.isclose(a5_data["rho"], rho), "threshold_ratio"].max()
    # Rounded, editable ranges; each column shares its scale across distributions.
    upper = np.ceil(max_ratio / 5) * 5 if max_ratio > 5 else np.ceil(max_ratio)
    ticks = [1, 10, 20] if upper > 20 else ([1, 5, 10] if upper > 5 else ([1, 3, 5] if upper > 3 else [1, 2, 3]))
    # Keep ratio=1 visibly inside every panel, even for the widest y ranges.
    y_top = upper * 1.03
    y_bottom = 1.0 - max(0.2, 0.08 * (y_top - 1.0))
    axes_a5[0, j].set_ylim(y_bottom, y_top)
    axes_a5[0, j].set_yticks(ticks)

fig_a5.subplots_adjust(left=0.095, right=0.95, bottom=0.273, top=0.932,
                        wspace=0.34, hspace=0.25)
a5_cax = fig_a5.add_axes([0.55, 0.105, 0.21, 0.019])
a5_cbar = fig_a5.colorbar(ScalarMappable(norm=a5_norm, cmap=a5_cmap),
                          cax=a5_cax, orientation="horizontal")
a5_cbar.set_ticks([0.1, 0.4, 0.7, 1.0])
a5_cbar.set_ticklabels(["0.1", "0.4", "0.7", "1"])
a5_cbar.ax.xaxis.set_minor_formatter(NullFormatter())
a5_cbar.ax.tick_params(labelsize=TWO_PANEL_STYLE["tick"])
a5_cbar.set_label(r"$\beta$", fontsize=TWO_PANEL_STYLE["label"])
fig_a5.legend(handles=[
    Line2D([], [], color="black", linewidth=1.2, alpha=0.8, label="Weak-signal prediction"),
], loc="center right", bbox_to_anchor=(0.52, 0.115), frameon=False,
    fontsize=TWO_PANEL_STYLE["legend"])
a5_output = PLOT_DIR / 'figure_18_finite_k_threshold_ratios.pdf'
save_pdf(fig_a5, 'figure_18_finite_k_threshold_ratios.pdf', dpi=300, bbox_inches='tight')
plt.show()
